# Compare CPU workers without changing the experiment
Upload to Colab, select a CPU runtime, and Run all using the same Google account. No GPU, new dependencies or Drive mount required.

After an interrupted older worker check, restart the Colab session before running this updated notebook. Interrupting a threaded fit may leave work active in the old session. This version prints each fit completion, a 20-second waiting message, and applies a 240-second per-task timeout. A timeout marks the comparison incomplete; it never changes scores or skips difficult rows. Thread cancellation cannot forcibly terminate an active native fit, so restart the session after a timeout too.

This compares **two threads versus two separate processes**, using the same three cached feature/horizon pairs and latest validation fold. Each phase has 21 fits; 42 diagnostic fits total. Same six settings, rows, purges, preprocessing and random seed. This is an execution test, not new feature research.

Both phases must reproduce saved pilot scores and setting selections. Their prediction digests must match for every inner and outer fit; independent tree traversal remains enabled. No change is adopted automatically. The full screen stays disabled.

Only existing research caches are used. The prior passing RUNTIME_PROFILE.json is required. Source metadata, runtime versions and cache hashes are checked. Missing/changed inputs stop the check; no source ZIP download fallback. No model binaries or row predictions are saved. Process workers receive only arrays and model code, not Drive access.

The timings include worker startup and data transfer. Uploads happen outside both timed phases. One fixed-order comparison is preliminary; it does not guarantee a speedup across all features. If only one CPU is usable, no fits run. If interrupted, at most 42 diagnostic fits restart; a matching completed result is reused.

The receipt is **WORKER_CHECK.json** in the existing output folder. Send the final printed summary. Earlier research and verification receipts stay unchanged.


In [ ]:
import hashlib, json
sources = {'reference': '"""BTCUSDT timing feature screen: pilot by default, full scope only after review.\nColab supplies CONFIG, RUNNER_SHA256 and DriveRangeReader in the notebook.\n"""\nimport io\nimport json\nimport re\nimport hashlib\nimport zipfile\nimport struct\nimport zlib\nimport time\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport numpy as np\nimport pandas as pd\n\n\ndef sha(data): return hashlib.sha256(data).hexdigest()\ndef signature(obj): return sha(json.dumps(obj,sort_keys=True,allow_nan=False).encode())\ndef utc(value): return pd.Timestamp(value,tz=\'UTC\')\n\n\ndef partition_masks(times, horizon, first, last):\n    start,stop=utc(first),utc(last)\n    ends=times+pd.Timedelta(minutes=horizon)\n    inner_start=start-pd.DateOffset(months=3)\n    return {\'inner_train\':(times<inner_start)&(ends<inner_start),\n            \'inner_valid\':(times>=inner_start)&(times<start)&(ends<start),\n            \'outer_train\':(times<start)&(ends<start),\n            \'outer_valid\':(times>=start)&(times<stop)}\n\n\ndef preprocess(x_train,x_predict,minimum):\n    observed=np.isfinite(x_train)\n    if observed.sum()<minimum or np.unique(x_train[observed]).size<2:\n        return None\n    median=float(np.median(x_train[observed]))\n    def transform(x):\n        missing=~np.isfinite(x)\n        return np.column_stack([np.where(missing,median,x),missing.astype(float)]).astype(np.float32)\n    return transform(x_train),transform(x_predict),median\n\n\ndef manual_predict(model,x):\n    # Independent traversal checks saved sklearn leaf predictions without another fit.\n    tree=model.tree_;nodes=np.zeros(len(x),dtype=np.int64)\n    for _ in range(model.get_depth()+1):\n        active=tree.children_left[nodes]!=-1\n        if not active.any():break\n        rows=np.flatnonzero(active);current=nodes[rows]\n        left=x[rows,tree.feature[current]]<=tree.threshold[current]\n        nodes[rows]=np.where(left,tree.children_left[current],tree.children_right[current])\n    assert (tree.children_left[nodes]==-1).all()\n    return tree.value[nodes,0,0]\n\n\ndef tree_fit(x_train,y_train,x_valid,y_valid,depth,leaf,minimum):\n    from sklearn.tree import DecisionTreeRegressor\n    start=time.perf_counter();prepared=preprocess(x_train,x_valid,minimum)\n    coverage={\'train_observed\':int(np.isfinite(x_train).sum()),\'train_rows\':len(x_train),\n              \'validation_observed\':int(np.isfinite(x_valid).sum()),\'validation_rows\':len(x_valid)}\n    if prepared is None:return {\'status\':\'INSUFFICIENT_TRAINING_COVERAGE_OR_VARIATION\',\'fit_count\':0,\'coverage\':coverage}\n    train,valid,median=prepared\n    model=DecisionTreeRegressor(criterion=\'absolute_error\',max_depth=depth,min_samples_leaf=leaf,random_state=48)\n    model.fit(train,y_train);prediction=model.predict(valid)\n    assert np.allclose(prediction,manual_predict(model,valid),rtol=0,atol=1e-10)\n    error=np.abs(prediction-y_valid);observed=np.isfinite(x_valid)\n    return {\'status\':\'PASS\',\'fit_count\':1,\'depth\':depth,\'min_samples_leaf\':leaf,\n            \'MAE_minutes\':float(error.mean()),\'absolute_error_sum\':float(error.sum()),\n            \'observed_rows_MAE\':float(error[observed].mean()) if observed.any() else None,\n            \'missing_rows_MAE\':float(error[~observed].mean()) if (~observed).any() else None,\n            \'coverage\':coverage,\'imputation_median\':median,\'tree_nodes\':int(model.tree_.node_count),\n            \'seconds\':time.perf_counter()-start,\'independent_tree_traversal\':\'PASS\'}\n\n\ndef pick_setting(results):\n    valid=[v for v in results if v[\'status\']==\'PASS\']\n    return min(valid,key=lambda v:(v[\'MAE_minutes\'],v[\'depth\'],-v[\'min_samples_leaf\'])) if valid else None\n\n\ndef self_test():\n    times=pd.date_range(\'2022-10-01\',\'2023-04-02\',freq=\'30min\',tz=\'UTC\')\n    masks=partition_masks(times,18000,\'2023-04-01\',\'2023-07-01\')\n    ends=times+pd.Timedelta(minutes=18000)\n    assert (ends[masks[\'inner_train\']]<utc(\'2023-01-01\')).all()\n    assert (ends[masks[\'inner_valid\']]<utc(\'2023-04-01\')).all()\n    assert (ends[masks[\'outer_train\']]<utc(\'2023-04-01\')).all()\n    prep=preprocess(np.array([0.,2.,np.nan]),np.array([10000.,np.nan]),2)\n    assert prep[2]==1 and prep[1][1,0]==1 and prep[1][1,1]==1\n    assert preprocess(np.zeros(10),np.ones(2),2) is None\n    x=np.repeat([0.,1.,2.,3.],100);y=np.where(x<2,1.,9.)\n    for depth,leaf in [(2,64),(3,64),(5,256)]:\n        score=tree_fit(x,y,np.array([0.,3.,np.nan]),np.array([1.,9.,1.]),depth,leaf,100)\n        assert score[\'status\']==\'PASS\' and score[\'coverage\'][\'validation_rows\']==3\n    tied=[{\'status\':\'PASS\',\'MAE_minutes\':1,\'depth\':3,\'min_samples_leaf\':64},\n          {\'status\':\'PASS\',\'MAE_minutes\':1,\'depth\':2,\'min_samples_leaf\':64},\n          {\'status\':\'PASS\',\'MAE_minutes\':1,\'depth\':2,\'min_samples_leaf\':256}]\n    assert pick_setting(tied)[\'min_samples_leaf\']==256\n    print(\'PASS: temporal purges, train-only imputation, missing-row preservation, deterministic ties and tree replay.\',flush=True)\n\n\nclass DriveStore:\n    def __init__(self,service,folder):self.service,self.folder=service,folder;self.ids={}\n    def find(self,name):\n        if name not in self.ids:\n            files=self.service.files().list(q=f"\'{self.folder}\' in parents and name=\'{name}\' and trashed=false",fields=\'files(id)\').execute()[\'files\']\n            assert len(files)<=1,\'Ambiguous output: \'+name\n            self.ids[name]=files[0][\'id\'] if files else None\n        return self.ids[name]\n    def read(self,name):\n        file=self.find(name)\n        return self.service.files().get_media(fileId=file).execute() if file else None\n    def write(self,name,data,mime=\'application/json\'):\n        from googleapiclient.http import MediaIoBaseUpload\n        assert len(data)<=128*1024**2,\'Compact cache limit exceeded\'\n        media=MediaIoBaseUpload(io.BytesIO(data),mimetype=mime,resumable=len(data)>5*1024**2)\n        file=self.find(name)\n        request=self.service.files().update(fileId=file,media_body=media,fields=\'id\') if file else self.service.files().create(body={\'name\':name,\'parents\':[self.folder]},media_body=media,fields=\'id\')\n        self.ids[name]=request.execute()[\'id\']\n    def json(self,name,obj):self.write(name,json.dumps(obj,indent=2,allow_nan=False).encode())\n\n\nclass RemoteArchive:\n    """Index once, read independent compressed members with bounded parallel I/O."""\n    def __init__(self,credentials,meta):\n        from google.auth.transport.requests import AuthorizedSession\n        self.credentials,self.meta=credentials,meta;self.local=threading.local();self.lock=threading.Lock();self.bytes=0\n        reader=DriveRangeReader(AuthorizedSession(credentials),meta[\'id\'],int(meta[\'size\']))\n        with zipfile.ZipFile(reader) as archive:self.members={x.filename:x for x in archive.infolist()}\n        self.bytes=reader.transferred\n    def read_range(self,start,length):\n        from google.auth.transport.requests import AuthorizedSession\n        if not hasattr(self.local,\'session\'):self.local.session=AuthorizedSession(self.credentials)\n        with self.lock:\n            assert self.bytes+length<=12*1024**3,\'Remote session byte budget exceeded\'\n            self.bytes+=length\n        reader=DriveRangeReader(self.local.session,self.meta[\'id\'],int(self.meta[\'size\']));reader.seek(start)\n        return reader.read(length)\n    def read(self,name):\n        info=self.members[name];header=self.read_range(info.header_offset,30)\n        fields=struct.unpack(\'<4s5H3I2H\',header);assert fields[0]==b\'PK\\x03\\x04\' and not fields[2]&1\n        raw=self.read_range(info.header_offset+30+fields[-2]+fields[-1],info.compress_size)\n        if info.compress_type==zipfile.ZIP_DEFLATED:data=zlib.decompress(raw,-15)\n        elif info.compress_type==zipfile.ZIP_STORED:data=raw\n        else:raise RuntimeError(\'Unsupported compression method; no full download fallback\')\n        assert len(data)==info.file_size and zlib.crc32(data)&0xffffffff==info.CRC\n        return data\n\n\ndef read_cache(store,name,expected):\n    raw=store.read(name)\n    if raw is None:return {\'identity\':expected,\'completed\':[],\'member_hashes\':{}},None,None\n    with np.load(io.BytesIO(raw),allow_pickle=False) as cache:\n        meta=json.loads(str(cache[\'metadata\'].item()));assert meta[\'identity\']==expected,\'Cache identity changed: \'+name\n        times=cache[\'times\'];values=cache[\'values\']\n        assert meta[\'array_sha256\']==sha(times.tobytes()+values.tobytes()),\'Cache content hash failed\'\n    return meta,times,values\n\n\ndef write_cache(store,name,meta,times,values):\n    meta[\'array_sha256\']=sha(times.tobytes()+values.tobytes())\n    buf=io.BytesIO();np.savez_compressed(buf,metadata=np.array(json.dumps(meta)),times=times,values=values)\n    store.write(name,buf.getvalue(),\'application/octet-stream\')\n\n\ndef load_target(archive,store,root,manifest,horizon,source,config):\n    import pyarrow.parquet as pq\n    name=f\'SCREEN_Y_{horizon}m.npz\';identity={\'type\':\'valid_time_high_research_v1\',\'source\':source,\'horizon\':horizon,\'origin_end\':config[\'origin_end_exclusive\'],\'cadence\':30}\n    meta,times,values=read_cache(store,name,identity)\n    ts=[] if times is None else [times];ys=[] if values is None else [values]\n    for month in config[\'months\']:\n        if month in meta[\'completed\']:continue\n        member=root+f\'y/horizon_minutes={horizon}/symbol=BTCUSDT/source_month={month}/part.parquet\'\n        raw=archive.read(member);assert sha(raw)==manifest[member]\n        frame=pq.read_table(io.BytesIO(raw),columns=[\'canonical_symbol\',\'decision_time\',\'horizon_minutes\',\'time_to_high_min\',\'label_status\']).to_pandas()\n        t=pd.DatetimeIndex(pd.to_datetime(frame.decision_time,utc=True));assert t.is_unique and frame.canonical_symbol.eq(\'BTCUSDT\').all() and frame.horizon_minutes.eq(horizon).all()\n        keep=(t.asi8%(30*60*10**9)==0)&(t< pd.Timestamp(config[\'origin_end_exclusive\']))&frame.label_status.eq(\'VALID\').to_numpy()\n        t=t[keep];y=frame.loc[keep,\'time_to_high_min\'].to_numpy(dtype=float)\n        assert np.isfinite(y).all() and ((y>=0)&(y<=horizon)).all()\n        assert ((t+pd.Timedelta(minutes=horizon))<pd.Timestamp(config[\'reserve_start\'])).all()\n        ts.append(t.asi8);ys.append(y);meta[\'completed\'].append(month);meta[\'member_hashes\'][member]=manifest[member]\n        write_cache(store,name,meta,np.concatenate(ts),np.concatenate(ys))\n        print(f\'Y {horizon}m: cached research month {month}\',flush=True)\n    times=np.concatenate(ts);values=np.concatenate(ys)\n    assert len(times)==len(np.unique(times)) and (np.diff(times)>0).all()\n    return pd.DatetimeIndex(times,tz=\'UTC\'),values,meta[\'array_sha256\']\n\n\ndef load_family(archive,store,family,columns,source,checksums,config,workers):\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    name=f\'SCREEN_X_F4Z_{family}.npz\'\n    identity={\'type\':\'f4z_research_cache_v1\',\'source\':source,\'columns\':columns,\'origin_end\':config[\'origin_end_exclusive\'],\n              \'cadence\':30,\'minimum_baseline_n\':1000,\'allowed_source_status\':[\'PROVISIONAL\']}\n    meta,times,values=read_cache(store,name,identity)\n    ts=[] if times is None else [times];xs=[] if values is None else [values]\n    members={}\n    for member in archive.members:\n        match=re.search(r\'/dates/[^/=]+=(\\d{4}-\\d{2}-\\d{2})/\',member)\n        if match and \'/F4Z/\' in member and f\'/family={family}/\' in member and \'/symbol=BTCUSDT/\' in member and member.endswith(\'.parquet\'):\n            if \'2022-04-22\'<=match[1]<=\'2024-06-30\':assert match[1] not in members;members[match[1]]=member\n    fields=[c.split(\'__\',2)[2] for c in columns]\n    wanted=[\'decision_time\',\'canonical_symbol\',\'available_after\']\n    for field in fields:wanted += [field,field[:-7]+\'__valid\',field[:-7]+\'__baseline_n\',field[:-7]+\'__scientific_status\']\n    def read_day(day):\n        member=members[day];raw=archive.read(member);assert sha(raw)==checksums[member]\n        table=pq.read_table(io.BytesIO(raw),columns=wanted)\n        t=pd.DatetimeIndex(pd.to_datetime(table[\'decision_time\'].to_pandas(),utc=True))\n        expected=pd.date_range(utc(day)+pd.Timedelta(minutes=1),periods=1440,freq=\'min\')\n        assert t.equals(expected) and set(table[\'canonical_symbol\'].to_pylist())=={\'BTCUSDT\'}\n        keep=(t.asi8%(30*60*10**9)==0)&(t<pd.Timestamp(config[\'origin_end_exclusive\']))\n        table=table.take(pa.array(np.flatnonzero(keep)));t=t[keep]\n        available=pd.DatetimeIndex(pd.to_datetime(table[\'available_after\'].to_pandas(),utc=True));causal=~available.isna()&(available<=t)\n        data=np.empty((len(t),len(fields)),dtype=np.float64)\n        for i,field in enumerate(fields):\n            base=field[:-7];v=table[field].cast(pa.float64()).to_numpy(zero_copy_only=False)\n            valid=table[base+\'__valid\'].to_pandas().fillna(False).to_numpy(dtype=bool)\n            n=table[base+\'__baseline_n\'].cast(pa.float64()).to_numpy(zero_copy_only=False)\n            status=np.array(table[base+\'__scientific_status\'].to_pylist())\n            good=causal&valid&np.isfinite(v)&np.isfinite(n)&(n>=1000)&(status==\'PROVISIONAL\')\n            data[:,i]=np.where(good,v,np.nan)\n        return t.asi8,data,member,checksums[member]\n    for month in config[\'months\']:\n        if month in meta[\'completed\']:continue\n        days=pd.date_range(max(utc(month+\'-01\'),utc(\'2022-04-22\')),utc(month+\'-01\')+pd.offsets.MonthEnd(0),freq=\'D\').strftime(\'%Y-%m-%d\').tolist()\n        assert set(days)<=set(members)\n        with ThreadPoolExecutor(max_workers=workers) as executor:\n            for count,(t,x,member,h) in enumerate(executor.map(read_day,days),1):\n                ts.append(t);xs.append(x);meta[\'member_hashes\'][member]=h\n                if count%5==0:print(f\'X {family} {month}: {count}/{len(days)} days read\',flush=True)\n        meta[\'completed\'].append(month);write_cache(store,name,meta,np.concatenate(ts),np.concatenate(xs))\n        print(f\'X {family}: cached research month {month}\',flush=True)\n    times=np.concatenate(ts);values=np.concatenate(xs)\n    assert len(times)==len(np.unique(times)) and (np.diff(times)>0).all()\n    return pd.DatetimeIndex(times,tz=\'UTC\'),values,meta[\'array_sha256\']\n\n\ndef check_baselines(times,y,horizon,baseline,config):\n    for i,(first,last) in enumerate(config[\'folds\']):\n        masks=partition_masks(times,horizon,first,last);train=masks[\'outer_train\'];valid=masks[\'outer_valid\']\n        expected=baseline[\'completed\'][str(horizon)][\'folds\'][i]\n        assert train.sum()==expected[\'train_rows\'] and valid.sum()==expected[\'validation_rows\']\n        constants=[np.median(y[train]),horizon/2,np.mean(y[train])]\n        for name,pred in zip([\'training_median_time_to_high\',\'horizon_midpoint\',\'training_mean_time_to_high\'],constants):\n            assert abs(float(np.abs(y[valid]-pred).mean())-expected[\'scores\'][name][\'MAE_minutes\'])<1e-8\n\n\ndef update_matrix(state,baseline,config):\n    for feature in config[\'candidate_features\']:\n        for horizon in config[\'horizons\']:\n            keys=[f\'{feature}|{horizon}|{i}\' for i in range(len(config[\'folds\']))]\n            if not all(key in state[\'folds\'] for key in keys):continue\n            folds=[state[\'folds\'][key] for key in keys]\n            count=sum(v[\'validation_rows\'] for v in folds);mae=sum(v[\'absolute_error_sum\'] for v in folds)/count\n            base=baseline[\'completed\'][str(horizon)][\'pooled\']\n            state[\'matrix\'][f\'{feature}|{horizon}\']={\'feature\':feature,\'horizon_minutes\':horizon,\'rows\':count,\'MAE_minutes\':mae,\n                \'MAE_percent_of_horizon\':100*mae/horizon,\'delta_MAE_vs_baselines\':{n:mae-v[\'MAE_minutes\'] for n,v in base.items()},\n                \'folds_using_tree\':sum(v[\'status\']==\'PASS\' for v in folds),\'folds_using_fallback\':sum(v[\'status\']!=\'PASS\' for v in folds),\n                \'status\':\'PROVISIONAL_DEVELOPMENT_SCREEN_NOT_TEST_PROMOTION\'}\n\n\ndef run(mode=\'pilot\'):\n    import re\n    import platform\n    import sklearn\n    import pyarrow\n    import os\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from threadpoolctl import threadpool_limits\n    assert mode in (\'pilot\',\'full\');self_test();auth.authenticate_user();credentials,_=google.auth.default()\n    service=build(\'drive\',\'v3\',credentials=credentials,cache_discovery=False);store=DriveStore(service,CONFIG[\'folder\'])\n    sources={k:service.files().get(fileId=v,fields=\'id,name,size,md5Checksum,modifiedTime\').execute() for k,v in CONFIG[\'source_ids\'].items()}\n    assert sources==CONFIG[\'audit_sources\'],\'Source archives changed since the feature audit\'\n    versions={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__,\'pyarrow\':pyarrow.__version__}\n    identity={\'config\':CONFIG,\'sources\':sources,\'runner_sha256\':RUNNER_SHA256,\'versions\':versions};sig=signature(identity)\n    prior=store.read(\'SCREEN_RESULTS.json\')\n    state=json.loads(prior) if prior else {\'signature\':sig,\'identity\':identity,\'jobs\':{},\'matrix\':{},\'status\':\'RUNNING_PILOT\',\'new_model_fits_total\':0,\'reserved_partitions_read\':0}\n    assert state[\'signature\']==sig,\'Screen identity changed; inspect results before replacement.\'\n    baseline=json.loads(store.read(\'BASELINE_RESULTS.json\'))\n    assert signature(baseline[\'identity\'])==baseline[\'signature\']==CONFIG[\'baseline_signature\']\n    assert baseline[\'identity\'][\'source\']==sources[\'Y\']\n    features=CONFIG[\'pilot_features\'] if mode==\'pilot\' else CONFIG[\'candidate_features\']\n    horizons=CONFIG[\'pilot_horizons\'] if mode==\'pilot\' else CONFIG[\'horizons\']\n    todo=[(f,h) for f in features for h in horizons if f\'{f}|{h}\' not in state[\'jobs\']]\n    if not todo:\n        state[\'status\']=\'PILOT_COMPLETE_REVIEW_BEFORE_FULL\' if mode==\'pilot\' else \'FULL_INDIVIDUAL_SCREEN_COMPLETE\'\n        store.json(\'SCREEN_RESULTS.json\',state)\n        print(\'Requested scope already complete. Reusing saved results; no archive reads.\');return\n    fit_workers=min(2,os.cpu_count() or 1);io_workers=min(4,os.cpu_count() or 1)\n    state[\'workers\']={\'fits\':fit_workers,\'remote_reads\':io_workers}\n    print(f\'Scope={mode}; {len(features)} features x {len(horizons)} horizons. Fits={fit_workers} workers; reads={io_workers} workers.\',flush=True)\n    folders=service.files().list(q=f"\'{CONFIG[\'folder\']}\' in parents and name=\'Timing_Screen_Fit_Scores\' and mimeType=\'application/vnd.google-apps.folder\' and trashed=false",fields=\'files(id)\').execute()[\'files\']\n    assert len(folders)<=1\n    fit_folder=folders[0][\'id\'] if folders else service.files().create(body={\'name\':\'Timing_Screen_Fit_Scores\',\'parents\':[CONFIG[\'folder\']],\'mimeType\':\'application/vnd.google-apps.folder\'},fields=\'id\').execute()[\'id\']\n    fit_store=DriveStore(service,fit_folder)\n    start=time.perf_counter();archives={k:RemoteArchive(credentials,v) for k,v in sources.items()}\n    yarchive=archives[\'Y\'];manifest_name=next(n for n in yarchive.members if n.endswith(\'/HANDOFF_2_MANIFEST.json\'))\n    root=manifest_name[:-len(\'HANDOFF_2_MANIFEST.json\')];manifest={root+e[\'path\']:e[\'sha256\'] for e in json.loads(yarchive.read(manifest_name))[\'files\']}\n    raw=archives[\'X\'].read(\'FILES.sha256\');checksums={}\n    for line in raw.decode().splitlines():\n        if line.strip():h,name=line.split(maxsplit=1);checksums[name.lstrip(\'*\')]=h\n    targets={}\n    target_start=time.perf_counter()\n    for horizon in sorted({h for f,h in todo}):\n        targets[horizon]=load_target(yarchive,store,root,manifest,horizon,sources[\'Y\'],CONFIG)\n        check_baselines(targets[horizon][0],targets[horizon][1],horizon,baseline,CONFIG)\n    state[\'target_loading_seconds_this_session\']=time.perf_counter()-target_start\n    def save():\n        state[\'session_elapsed_seconds\']=time.perf_counter()-start\n        state[\'remote_bytes_this_session\']={k:v.bytes for k,v in archives.items()}\n        state[\'new_model_fits_total\']=sum(v[\'fit_count\'] for v in state[\'jobs\'].values())\n        state[\'summed_fit_seconds\']=sum(v.get(\'fit_seconds\',0) for v in state[\'jobs\'].values())\n        try:\n            import resource\n            state[\'peak_process_RSS_MiB\']=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024\n        except ImportError:state[\'peak_process_RSS_MiB\']=None\n        store.json(\'SCREEN_RESULTS.json\',state)\n    for family in sorted({f.split(\'__\')[1] for f,h in todo}):\n        columns=[c for c in CONFIG[\'candidate_features\'] if c.split(\'__\')[1]==family]\n        family_start=time.perf_counter()\n        xt,xdata,xhash=load_family(archives[\'X\'],store,family,columns,sources[\'X\'],checksums,CONFIG,io_workers)\n        state.setdefault(\'family_loading_seconds_this_session\',{})[family]=time.perf_counter()-family_start\n        for feature in [f for f in features if f.split(\'__\')[1]==family]:\n            xcol=xdata[:,columns.index(feature)]\n            for horizon in horizons:\n                if (feature,horizon) not in todo:continue\n                times,y,yhash=targets[horizon];indices=xt.get_indexer(times);assert (indices>=0).all();x=xcol[indices]\n                job_id=feature+\'|\'+str(horizon);job_name=\'job_\'+sha(job_id.encode())[:24]+\'.json\'\n                saved_job=fit_store.read(job_name)\n                job=json.loads(saved_job) if saved_job else {\'screen_signature\':sig,\'feature\':feature,\'horizon_minutes\':horizon,\'X_cache_sha256\':xhash,\'Y_cache_sha256\':yhash,\'fits\':{},\'folds\':{},\'matrix\':{}}\n                assert job[\'screen_signature\']==sig and job[\'feature\']==feature and job[\'horizon_minutes\']==horizon\n                assert job[\'X_cache_sha256\']==xhash and job[\'Y_cache_sha256\']==yhash\n                def save_job():fit_store.json(job_name,job)\n                for fold,(first,last) in enumerate(CONFIG[\'folds\']):\n                    key=f\'{feature}|{horizon}|{fold}\'\n                    if key in job[\'folds\']:continue\n                    masks=partition_masks(times,horizon,first,last);it,iv=masks[\'inner_train\'],masks[\'inner_valid\'];ot,ov=masks[\'outer_train\'],masks[\'outer_valid\']\n                    assert it.sum() and iv.sum() and ot.sum() and ov.sum()\n                    pending=[];inner=[]\n                    with threadpool_limits(limits=1),ThreadPoolExecutor(max_workers=fit_workers) as executor:\n                        for depth,leaf in CONFIG[\'grid\']:\n                            fit_key=f\'{key}|inner|{depth}|{leaf}\'\n                            if fit_key in job[\'fits\']:inner.append(job[\'fits\'][fit_key]);continue\n                            future=executor.submit(tree_fit,x[it],y[it],x[iv],y[iv],depth,leaf,CONFIG[\'minimum_observed_training\'])\n                            pending.append((future,fit_key))\n                        lookup={future:name for future,name in pending}\n                        for future in as_completed(lookup):\n                            score=future.result();job[\'fits\'][lookup[future]]=score;inner.append(score);save_job()\n                            print(f\'{horizon}m {feature} fold {fold+1}: inner setting saved\',flush=True)\n                    selected=pick_setting(inner);fit_key=key+\'|outer\'\n                    if fit_key not in job[\'fits\']:\n                        outer=tree_fit(x[ot],y[ot],x[ov],y[ov],selected[\'depth\'],selected[\'min_samples_leaf\'],CONFIG[\'minimum_observed_training\']) if selected else {\'status\':\'INNER_FEATURE_INELIGIBLE\',\'fit_count\':0}\n                        if outer[\'status\']!=\'PASS\':\n                            pred=float(np.median(y[ot]));errors=np.abs(y[ov]-pred)\n                            outer.update({\'fallback\':\'training_median_time_to_high\',\'MAE_minutes\':float(errors.mean()),\'absolute_error_sum\':float(errors.sum())})\n                        job[\'fits\'][fit_key]=outer;save_job()\n                    outer=job[\'fits\'][fit_key]\n                    job[\'folds\'][key]={\'feature\':feature,\'horizon_minutes\':horizon,\'fold\':fold+1,\n                        \'train_rows\':int(ot.sum()),\'validation_rows\':int(ov.sum()),\'inner_train_rows\':int(it.sum()),\'inner_validation_rows\':int(iv.sum()),\n                        \'inner_latest_training_label_end\':str((times[it]+pd.Timedelta(minutes=horizon)).max()),\n                        \'inner_latest_validation_label_end\':str((times[iv]+pd.Timedelta(minutes=horizon)).max()),\n                        \'outer_latest_training_label_end\':str((times[ot]+pd.Timedelta(minutes=horizon)).max()),\n                        \'selected\':None if selected is None else {\'depth\':selected[\'depth\'],\'min_samples_leaf\':selected[\'min_samples_leaf\']},\n                        \'MAE_minutes\':outer[\'MAE_minutes\'],\'absolute_error_sum\':outer[\'absolute_error_sum\'],\n                        \'status\':outer[\'status\'],\'X_cache_sha256\':xhash,\'Y_cache_sha256\':yhash,\'baseline_reproduction\':\'PASS\'}\n                    save_job();print(f\'{horizon}m {feature} fold {fold+1}: outer MAE={outer["MAE_minutes"]:.3f}\',flush=True)\n                update_matrix(job,baseline,dict(CONFIG,candidate_features=[feature],horizons=[horizon]))\n                save_job();state[\'matrix\'].update(job[\'matrix\'])\n                state[\'jobs\'][job_id]={\'fit_count\':sum(v.get(\'fit_count\',0) for v in job[\'fits\'].values()),\'fit_seconds\':sum(v.get(\'seconds\',0) for v in job[\'fits\'].values()),\'score_file\':job_name,\'score_file_id\':fit_store.ids[job_name],\n                                      \'score_sha256\':signature(job),\'status\':\'COMPLETE\'}\n                save()\n        del xdata\n    for k,v in CONFIG[\'source_ids\'].items():assert service.files().get(fileId=v,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()==sources[k]\n    state[\'status\']=\'PILOT_COMPLETE_REVIEW_BEFORE_FULL\' if mode==\'pilot\' else \'FULL_INDIVIDUAL_SCREEN_COMPLETE\';save()\n    print(\'\\nDONE: \'+state[\'status\'])\n    print(\'Horizon | feature | MAE minutes | delta vs lowest baseline (negative is better)\')\n    for feature in features:\n        for horizon in horizons:\n            row=state[\'matrix\'][f\'{feature}|{horizon}\'];print(horizon,feature,round(row[\'MAE_minutes\'],3),round(max(row[\'delta_MAE_vs_baselines\'].values()),3))\n    print(\'Send only this folder link: https://drive.google.com/drive/folders/\'+CONFIG[\'folder\'])\n', 'optimized': '"""Bounded execution and cache-only equivalence benchmark. No full-screen entry point.\n\nREF is the frozen screen_runner namespace, supplied by the notebook/test.\n"""\nimport json\nimport time\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nfrom collections import deque\nimport numpy as np\n\n\ndef scientific(score):\n    return {k: v for k, v in score.items() if k != \'seconds\'}\n\n\nclass OrderedWriter:\n    """One writer owns its store; serialize before queuing, bound pending snapshots."""\n    def __init__(self, write, capacity=2):\n        self.write = write\n        self.capacity = capacity\n        self.pool = ThreadPoolExecutor(max_workers=1)\n        self.pending = deque()\n        self.lock = threading.Lock()\n\n    def submit(self, name, value):\n        raw = json.dumps(value, allow_nan=False).encode()\n        with self.lock:\n            while len(self.pending) >= self.capacity:\n                self.pending.popleft().result()\n            self.pending.append(self.pool.submit(self.write, name, raw))\n\n    def close(self):\n        try:\n            while self.pending:\n                self.pending.popleft().result()\n        finally:\n            self.pool.shutdown(wait=True)\n\n\ndef prepare(x_train, y_train, x_valid, y_valid, minimum):\n    return (REF[\'preprocess\'](x_train, x_valid, minimum), y_train, y_valid,\n            np.isfinite(x_valid),\n            {\'train_observed\': int(np.isfinite(x_train).sum()), \'train_rows\': len(x_train),\n             \'validation_observed\': int(np.isfinite(x_valid).sum()), \'validation_rows\': len(x_valid)})\n\n\ndef prepared_fit(data, depth, leaf):\n    from sklearn.tree import DecisionTreeRegressor\n    start = time.perf_counter()\n    prepared, y_train, y_valid, observed, coverage = data\n    if prepared is None:\n        return {\'status\': \'INSUFFICIENT_TRAINING_COVERAGE_OR_VARIATION\',\n                \'fit_count\': 0, \'coverage\': coverage}, None\n    train, valid, median = prepared\n    model = DecisionTreeRegressor(criterion=\'absolute_error\', max_depth=depth,\n                                 min_samples_leaf=leaf, random_state=48)\n    model.fit(train, y_train)\n    prediction = model.predict(valid)\n    assert np.array_equal(prediction, REF[\'manual_predict\'](model, valid))\n    error = np.abs(prediction-y_valid)\n    score = {\'status\': \'PASS\', \'fit_count\': 1, \'depth\': depth, \'min_samples_leaf\': leaf,\n             \'MAE_minutes\': float(error.mean()), \'absolute_error_sum\': float(error.sum()),\n             \'observed_rows_MAE\': float(error[observed].mean()) if observed.any() else None,\n             \'missing_rows_MAE\': float(error[~observed].mean()) if (~observed).any() else None,\n             \'coverage\': coverage, \'imputation_median\': median,\n             \'tree_nodes\': int(model.tree_.node_count), \'seconds\': time.perf_counter()-start,\n             \'independent_tree_traversal\': \'PASS\'}\n    return score, prediction\n\n\ndef fold_task(x, y, masks, config, optimized, emit=lambda key, score: None):\n    """One fold per task; pool has at most two tasks, including outer fits.\n\n    Predictions are kept only in memory for equivalence checks, never checkpointed.\n    """\n    it, iv, ot, ov = [masks[k] for k in (\'inner_train\', \'inner_valid\', \'outer_train\', \'outer_valid\')]\n    minimum = config[\'minimum_observed_training\']\n    fits, predictions = {}, {}\n    data = prepare(x[it], y[it], x[iv], y[iv], minimum) if optimized else None\n    def fit(train, valid, depth, leaf, prepared):\n        if optimized:\n            return prepared_fit(prepared, depth, leaf)\n        # Capture the frozen implementation\'s independent prediction traversal.\n        local = {}\n        original = REF[\'manual_predict\']\n        def capture(model, values):\n            prediction = original(model, values)\n            local[\'prediction\'] = prediction\n            return prediction\n        import types\n        namespace = dict(REF[\'tree_fit\'].__globals__, manual_predict=capture)\n        reference = types.FunctionType(REF[\'tree_fit\'].__code__, namespace)\n        score = reference(x[train], y[train], x[valid], y[valid], depth, leaf, minimum)\n        return score, local.get(\'prediction\')\n    for depth, leaf in config[\'grid\']:\n        key = f\'inner|{depth}|{leaf}\'\n        fits[key], predictions[key] = fit(it, iv, depth, leaf, data)\n        emit(key, fits[key])\n    selected = REF[\'pick_setting\'](list(fits.values()))\n    data = prepare(x[ot], y[ot], x[ov], y[ov], minimum) if optimized and selected else None\n    outer, prediction = fit(ot, ov, selected[\'depth\'], selected[\'min_samples_leaf\'], data) if selected else ({\'status\': \'INNER_FEATURE_INELIGIBLE\', \'fit_count\': 0}, None)\n    if outer[\'status\'] != \'PASS\':\n        prediction = np.full(int(ov.sum()), float(np.median(y[ot])))\n        error = np.abs(y[ov]-prediction)\n        outer.update(fallback=\'training_median_time_to_high\', MAE_minutes=float(error.mean()), absolute_error_sum=float(error.sum()))\n    fits[\'outer\'], predictions[\'outer\'] = outer, prediction\n    emit(\'outer\', outer)\n    return {\'fits\': fits, \'predictions\': predictions,\n            \'selected\': None if selected is None else [selected[\'depth\'], selected[\'min_samples_leaf\']]}\n\n\ndef assert_equal(a, b):\n    assert a[\'selected\'] == b[\'selected\']\n    assert a[\'fits\'].keys() == b[\'fits\'].keys()\n    for key in a[\'fits\']:\n        assert scientific(a[\'fits\'][key]) == scientific(b[\'fits\'][key]), key\n        left, right = a[\'predictions\'][key], b[\'predictions\'][key]\n        assert (left is None and right is None) or np.array_equal(left, right), key\n\n\ndef run_benchmark(config, reference_hash, optimized_hash, expected_pilot_identity):\n    import os\n    import platform\n    import pandas as pd\n    import sklearn\n    import pyarrow\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from threadpoolctl import threadpool_limits\n    auth.authenticate_user()\n    credentials, _ = google.auth.default()\n    service = build(\'drive\', \'v3\', credentials=credentials, cache_discovery=False)\n    store = REF[\'DriveStore\'](service, config[\'folder\'])\n    saved = json.loads(store.read(\'SCREEN_RESULTS.json\'))\n    versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,\n                    sklearn=sklearn.__version__, pyarrow=pyarrow.__version__)\n    assert saved[\'identity\'] == expected_pilot_identity, \'Pilot identity changed\'\n    assert saved[\'identity\'][\'config\'] == config and saved[\'identity\'][\'sources\'] == config[\'audit_sources\']\n    assert saved[\'identity\'][\'versions\'] == versions, \'Runtime changed; stop for review. Do not replace pilot.\'\n    assert REF[\'signature\'](saved[\'identity\']) == saved[\'signature\']\n    prior = store.read(\'THROUGHPUT_CHECK.json\')\n    if prior:\n        receipt = json.loads(prior)\n        if (receipt.get(\'status\') == \'PASS_EXACT_PREDICTIONS_SCORES_SELECTION_AND_SAVED_PILOT\'\n                and receipt.get(\'reference_sha256\') == reference_hash\n                and receipt.get(\'optimized_sha256\') == optimized_hash\n                and receipt.get(\'screen_signature\') == saved[\'signature\']):\n            print(\'This verification already passed. Reusing receipt; no fits repeated.\')\n            return\n    for key, file_id in config[\'source_ids\'].items():\n        current = service.files().get(fileId=file_id, fields=\'id,name,size,md5Checksum,modifiedTime\').execute()\n        assert current == config[\'audit_sources\'][key]\n    baseline = json.loads(store.read(\'BASELINE_RESULTS.json\'))\n    assert REF[\'signature\'](baseline[\'identity\']) == baseline[\'signature\'] == config[\'baseline_signature\']\n    columns = [f for f in config[\'candidate_features\'] if f.split(\'__\')[1] == \'flow\']\n    identity = dict(type=\'f4z_research_cache_v1\', source=config[\'audit_sources\'][\'X\'], columns=columns,\n                    origin_end=config[\'origin_end_exclusive\'], cadence=30, minimum_baseline_n=1000,\n                    allowed_source_status=[\'PROVISIONAL\'])\n    meta, xt, matrix = REF[\'read_cache\'](store, \'SCREEN_X_F4Z_flow.npz\', identity)\n    assert meta[\'completed\'] == config[\'months\'] and xt is not None\n    xt = pd.DatetimeIndex(xt, tz=\'UTC\')\n    assert xt.is_unique and xt.is_monotonic_increasing and (xt < pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n    tasks, existing = [], []\n    # Fixed before seeing scores: each pilot feature paired with one pilot horizon,\n    # latest outer fold (largest training population); 21 fits per implementation.\n    for feature, horizon in zip(config[\'pilot_features\'], config[\'pilot_horizons\']):\n        identity = dict(type=\'valid_time_high_research_v1\', source=config[\'audit_sources\'][\'Y\'],\n                        horizon=horizon, origin_end=config[\'origin_end_exclusive\'], cadence=30)\n        ym, times, y = REF[\'read_cache\'](store, f\'SCREEN_Y_{horizon}m.npz\', identity)\n        assert ym[\'completed\'] == config[\'months\'] and times is not None\n        times = pd.DatetimeIndex(times, tz=\'UTC\')\n        assert times.is_unique and times.is_monotonic_increasing\n        assert (times < pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n        assert ((times+pd.Timedelta(minutes=horizon)) < pd.Timestamp(config[\'reserve_start\'])).all()\n        REF[\'check_baselines\'](times, y, horizon, baseline, config)\n        indices = xt.get_indexer(times)\n        assert (indices >= 0).all()\n        masks = REF[\'partition_masks\'](times, horizon, *config[\'folds\'][-1])\n        tasks.append((matrix[indices, columns.index(feature)], y, masks, config))\n        job_id = feature+\'|\'+str(horizon)\n        info = saved[\'jobs\'][job_id]\n        job = json.loads(service.files().get_media(fileId=info[\'score_file_id\']).execute())\n        assert REF[\'signature\'](job) == info[\'score_sha256\']\n        assert job[\'screen_signature\'] == saved[\'signature\']\n        assert job[\'X_cache_sha256\'] == meta[\'array_sha256\'] and job[\'Y_cache_sha256\'] == ym[\'array_sha256\']\n        existing.append((job_id, job))\n    output = {\'status\': \'RUNNING_EQUIVALENCE_ONLY\', \'reference_sha256\': reference_hash,\n              \'optimized_sha256\': optimized_hash, \'screen_signature\': saved[\'signature\'],\n              \'pilot_declared_runner_sha256\': saved[\'identity\'][\'runner_sha256\'],\n              \'reference_hash_matches_pilot_declaration\': reference_hash == saved[\'identity\'][\'runner_sha256\'],\n              \'versions\': versions, \'scope\': \'three fixed pilot pairs, outer fold 5, 42 total fits\',\n              \'full_run_authorized\': False, \'scores\': {}, \'timings\': {}}\n    output[\'fit_workers\'] = min(2, os.cpu_count() or 1)\n    # Dedicated service is owned exclusively by the single writer thread.\n    writer_store = REF[\'DriveStore\'](build(\'drive\', \'v3\', credentials=credentials, cache_discovery=False), config[\'folder\'])\n    writer = OrderedWriter(writer_store.write)\n    lock = threading.Lock()\n    def emit(task_id, key, score):\n        with lock:\n            output[\'scores\'][f\'{task_id}|{key}\'] = score\n            writer.submit(\'THROUGHPUT_CHECK.json\', output)\n    results = {}\n    try:\n        with threadpool_limits(limits=1):\n            for optimized in (False, True):\n                label = \'optimized\' if optimized else \'reference\'\n                start = time.perf_counter()\n                with ThreadPoolExecutor(max_workers=output[\'fit_workers\']) as pool:\n                    futures = [pool.submit(fold_task, *task, optimized,\n                               lambda key, score, i=i: emit(f\'{label}|{i}\', key, score))\n                               for i, task in enumerate(tasks)]\n                    results[label] = [f.result() for f in futures]\n                output[\'timings\'][label+\'_compute_and_enqueue_seconds\'] = time.perf_counter()-start\n                # Barrier: include all remote checkpoint completion in each timing.\n                writer.close()\n                output[\'timings\'][label+\'_including_upload_seconds\'] = time.perf_counter()-start\n                writer = OrderedWriter(writer_store.write)\n                print(label+\' phase complete\', flush=True)\n        for i, (a, b) in enumerate(zip(results[\'reference\'], results[\'optimized\'])):\n            assert_equal(a, b)\n            job_id, job = existing[i]\n            for key, score in a[\'fits\'].items():\n                assert scientific(score) == scientific(job[\'fits\'][f\'{job_id}|4|{key}\'])\n        output[\'status\'] = \'PASS_EXACT_PREDICTIONS_SCORES_SELECTION_AND_SAVED_PILOT\'\n        output[\'limitations\'] = [\'Three fixed cases only; not full-universe equivalence or a runtime guarantee.\',\n                                \'Both phases use bounded fold scheduling and asynchronous writes; timing isolates preparation reuse, not total speedup over the old scheduler.\',\n                                \'No automatic checkpoint migration or full launch. Review this receipt first.\',\n                                \'Current reference content hash differs from historical pilot declaration; saved-score replay is required, not identity equivalence.\',\n                                \'An interrupted verification may repeat at most 42 fits; a matching PASS receipt is reused.\']\n        writer.submit(\'THROUGHPUT_CHECK.json\', output)\n    except BaseException:\n        output[\'status\'] = \'FAILED_OR_INTERRUPTED_NOT_APPROVED\'\n        # Drain earlier snapshots before attempting a final failure receipt.\n        # A failed upload must not be hidden by a secondary shutdown error.\n        try:\n            writer.close()\n        except Exception:\n            pass\n        writer = OrderedWriter(writer_store.write)\n        try:\n            writer.submit(\'THROUGHPUT_CHECK.json\', output)\n            writer.close()\n        except Exception:\n            print(\'Failure receipt could not be uploaded; this run is NOT verified.\')\n        raise\n    finally:\n        writer.close()\n    print(output[\'status\'])\n    print(json.dumps(output[\'timings\'], indent=2))\n    print(\'Saved THROUGHPUT_CHECK.json. Full screen remains disabled.\')\n', 'profile': '"""Cache-only timing diagnostic. REF and OPT are embedded notebook namespaces."""\nimport json\nimport os\nimport platform\nimport threading\nimport time\nimport types\nfrom concurrent.futures import ThreadPoolExecutor\nimport numpy as np\nimport pandas as pd\n\n\ndef measured_fold(task, emit):\n    # Per-task timers: no interpreter-wide profiler/tool IDs or global monkeypatches.\n    timing = dict.fromkeys([\'preparation_seconds\', \'model_fit_seconds\', \'prediction_seconds\',\n                           \'independent_traversal_seconds\', \'fit_and_scoring_seconds\',\n                           \'checkpoint_emit_seconds\'], 0.0)\n    def timed(name, function, *args):\n        start = time.perf_counter()\n        try:\n            return function(*args)\n        finally:\n            timing[name] += time.perf_counter()-start\n    def prepared_fit(data, depth, leaf):\n        from sklearn.tree import DecisionTreeRegressor\n        start = time.perf_counter()\n        prepared, y_train, y_valid, observed, coverage = data\n        if prepared is None:\n            return {\'status\':\'INSUFFICIENT_TRAINING_COVERAGE_OR_VARIATION\',\n                    \'fit_count\':0, \'coverage\':coverage}, None\n        train, valid, median = prepared\n        model = DecisionTreeRegressor(criterion=\'absolute_error\', max_depth=depth,\n                                      min_samples_leaf=leaf, random_state=48)\n        timed(\'model_fit_seconds\', model.fit, train, y_train)\n        prediction = timed(\'prediction_seconds\', model.predict, valid)\n        independent = timed(\'independent_traversal_seconds\', REF[\'manual_predict\'], model, valid)\n        assert np.array_equal(prediction, independent)\n        error = np.abs(prediction-y_valid)\n        return {\'status\':\'PASS\', \'fit_count\':1, \'depth\':depth, \'min_samples_leaf\':leaf,\n                \'MAE_minutes\':float(error.mean()), \'absolute_error_sum\':float(error.sum()),\n                \'observed_rows_MAE\':float(error[observed].mean()) if observed.any() else None,\n                \'missing_rows_MAE\':float(error[~observed].mean()) if (~observed).any() else None,\n                \'coverage\':coverage, \'imputation_median\':median,\n                \'tree_nodes\':int(model.tree_.node_count), \'seconds\':time.perf_counter()-start,\n                \'independent_tree_traversal\':\'PASS\'}, prediction\n    namespace = dict(OPT[\'fold_task\'].__globals__,\n                     prepare=lambda *args: timed(\'preparation_seconds\', OPT[\'prepare\'], *args),\n                     prepared_fit=lambda *args: timed(\'fit_and_scoring_seconds\', prepared_fit, *args))\n    fold = types.FunctionType(OPT[\'fold_task\'].__code__, namespace)\n    start = time.perf_counter()\n    result = fold(*task, True, lambda *args: timed(\'checkpoint_emit_seconds\', emit, *args))\n    timing[\'fold_wall_seconds\'] = time.perf_counter()-start\n    return result, timing\n\n\ndef load_tasks(store, service, config, saved):\n    baseline = json.loads(store.read(\'BASELINE_RESULTS.json\'))\n    assert REF[\'signature\'](baseline[\'identity\']) == baseline[\'signature\'] == config[\'baseline_signature\']\n    assert baseline[\'identity\'][\'source\'] == config[\'audit_sources\'][\'Y\']\n    columns = [f for f in config[\'candidate_features\'] if f.split(\'__\')[1] == \'flow\']\n    identity = dict(type=\'f4z_research_cache_v1\', source=config[\'audit_sources\'][\'X\'], columns=columns,\n                    origin_end=config[\'origin_end_exclusive\'], cadence=30,\n                    minimum_baseline_n=1000, allowed_source_status=[\'PROVISIONAL\'])\n    meta, xt, matrix = REF[\'read_cache\'](store, \'SCREEN_X_F4Z_flow.npz\', identity)\n    assert meta[\'completed\'] == config[\'months\'] and xt is not None, \'Complete existing flow cache required\'\n    xt = pd.DatetimeIndex(xt, tz=\'UTC\')\n    assert xt.is_unique and xt.is_monotonic_increasing\n    assert (xt < pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n    tasks, expected, cases = [], [], []\n    for feature, horizon in zip(config[\'pilot_features\'], config[\'pilot_horizons\']):\n        identity = dict(type=\'valid_time_high_research_v1\', source=config[\'audit_sources\'][\'Y\'],\n                        horizon=horizon, origin_end=config[\'origin_end_exclusive\'], cadence=30)\n        ym, times, y = REF[\'read_cache\'](store, f\'SCREEN_Y_{horizon}m.npz\', identity)\n        assert ym[\'completed\'] == config[\'months\'] and times is not None, \'Complete existing Y cache required\'\n        times = pd.DatetimeIndex(times, tz=\'UTC\')\n        assert times.is_unique and times.is_monotonic_increasing\n        assert (times < pd.Timestamp(config[\'origin_end_exclusive\'])).all()\n        assert ((times+pd.Timedelta(minutes=horizon)) < pd.Timestamp(config[\'reserve_start\'])).all()\n        assert np.isfinite(y).all() and ((y >= 0) & (y <= horizon)).all()\n        REF[\'check_baselines\'](times, y, horizon, baseline, config)\n        indices = xt.get_indexer(times)\n        assert (indices >= 0).all()\n        tasks.append((matrix[indices, columns.index(feature)], y,\n                      REF[\'partition_masks\'](times, horizon, *config[\'folds\'][-1]), config))\n        job_id = feature+\'|\'+str(horizon)\n        info = saved[\'jobs\'][job_id]\n        job = json.loads(service.files().get_media(fileId=info[\'score_file_id\']).execute())\n        assert REF[\'signature\'](job) == info[\'score_sha256\'] and job[\'screen_signature\'] == saved[\'signature\']\n        assert job[\'X_cache_sha256\'] == meta[\'array_sha256\'] and job[\'Y_cache_sha256\'] == ym[\'array_sha256\']\n        expected.append({key: job[\'fits\'][f\'{job_id}|4|{key}\']\n                         for key in [f\'inner|{d}|{l}\' for d, l in config[\'grid\']]+[\'outer\']})\n        cases.append({\'feature\':feature, \'horizon_minutes\':horizon, \'outer_fold\':5})\n    return tasks, expected, cases\n\n\ndef run_profile(config, expected_identity, hashes):\n    import sklearn\n    import pyarrow\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from threadpoolctl import threadpool_limits\n    auth.authenticate_user()\n    credentials, _ = google.auth.default()\n    service = build(\'drive\', \'v3\', credentials=credentials, cache_discovery=False)\n    store = REF[\'DriveStore\'](service, config[\'folder\'])\n    saved = json.loads(store.read(\'SCREEN_RESULTS.json\'))\n    versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,\n                    sklearn=sklearn.__version__, pyarrow=pyarrow.__version__)\n    assert saved[\'identity\'] == expected_identity and REF[\'signature\'](expected_identity) == saved[\'signature\']\n    assert expected_identity[\'config\'] == config and expected_identity[\'versions\'] == versions, \'Runtime/config changed; stop for review\'\n    prior_raw = store.read(\'THROUGHPUT_CHECK.json\')\n    assert prior_raw, \'Completed equivalence receipt required\'\n    prior = json.loads(prior_raw)\n    assert prior[\'status\'] == \'PASS_EXACT_PREDICTIONS_SCORES_SELECTION_AND_SAVED_PILOT\'\n    assert prior[\'screen_signature\'] == saved[\'signature\']\n    assert prior[\'reference_sha256\'] == hashes[\'reference\'] and prior[\'optimized_sha256\'] == hashes[\'optimized\']\n    for key, file_id in config[\'source_ids\'].items():\n        assert service.files().get(fileId=file_id, fields=\'id,name,size,md5Checksum,modifiedTime\').execute() == config[\'audit_sources\'][key]\n    identity = {\'hashes\':hashes, \'pilot_signature\':saved[\'signature\'],\n                \'equivalence_sha256\':REF[\'sha\'](prior_raw), \'versions\':versions, \'scope\':\'three fixed pairs, outer fold 5\'}\n    signature = REF[\'signature\'](identity)\n    existing = store.read(\'RUNTIME_PROFILE.json\')\n    if existing:\n        previous = json.loads(existing)\n        if previous.get(\'signature\') == signature and previous.get(\'status\') == \'PASS_PROFILE_AND_SAVED_SCORES\':\n            print(\'Reusing completed profile; no fits repeated.\')\n            print(json.dumps(previous[\'summary\'], indent=2))\n            return\n    start = time.perf_counter()\n    tasks, expected, cases = load_tasks(store, service, config, saved)\n    load_seconds = time.perf_counter()-start\n    output = {\'identity\':identity, \'signature\':signature, \'status\':\'RUNNING_PROFILE_ONLY\',\n              \'cases\':cases, \'fits\':{}, \'timings\':{}, \'cache_load_and_validation_seconds\':load_seconds,\n              \'full_screen_authorized\':False}\n    # Writer owns its own HTTP transport; never share it with fit threads.\n    writer_store = REF[\'DriveStore\'](build(\'drive\', \'v3\', credentials=credentials, cache_discovery=False), config[\'folder\'])\n    uploads, submission = [], []\n    def upload(name, raw):\n        start = time.perf_counter()\n        writer_store.write(name, raw)\n        uploads.append(time.perf_counter()-start)\n    writer = OPT[\'OrderedWriter\'](upload, capacity=2)\n    lock = threading.Lock()\n    def emit(case, key, score):\n        start = time.perf_counter()\n        with lock:\n            assert OPT[\'scientific\'](score) == OPT[\'scientific\'](expected[case][key]), \'Saved pilot score mismatch\'\n            output[\'fits\'][f\'{case}|{key}\'] = score\n            writer.submit(\'RUNTIME_PROFILE.json\', output)\n        submission.append(time.perf_counter()-start)\n    workers = min(2, os.cpu_count() or 1)\n    cpu_start, start = time.process_time(), time.perf_counter()\n    try:\n        with threadpool_limits(limits=1), ThreadPoolExecutor(max_workers=workers) as pool:\n            futures = [pool.submit(measured_fold, task, lambda key, score, i=i: emit(i, key, score))\n                       for i, task in enumerate(tasks)]\n            for i, future in enumerate(futures):\n                result, timing = future.result()\n                # Avoid mutating checkpoint snapshots while another thread serializes.\n                with lock:\n                    output[\'timings\'][str(i)] = timing\n                del result  # No row predictions or models retained on disk.\n                print(f\'Profile case {i+1}/3 complete\', flush=True)\n        before_drain = time.perf_counter()-start\n        writer.close()\n        wall, cpu = time.perf_counter()-start, time.process_time()-cpu_start\n    except BaseException:\n        try:\n            writer.close()\n        except Exception:\n            pass\n        # Partial output remains RUNNING, never a verified result; rerun is bounded.\n        raise\n    summed = {key:sum(t[key] for t in output[\'timings\'].values()) for key in next(iter(output[\'timings\'].values()))}\n    output[\'summary\'] = {\'fits\':len(output[\'fits\']), \'fit_workers\':workers,\n                         \'wall_including_checkpoints_seconds\':wall,\n                         \'final_checkpoint_drain_seconds\':wall-before_drain,\n                         \'process_cpu_seconds\':cpu, \'average_cpu_cores_used\':cpu/wall,\n                         \'summed_task_timings_seconds\':summed,\n                         \'summed_checkpoint_submit_seconds\':sum(submission),\n                         \'summed_upload_seconds\':sum(uploads), \'checkpoint_upload_count\':len(uploads)}\n    output[\'limitations\'] = [\'Task durations overlap across workers; do not add them to wall time.\',\n                            \'Fit-and-scoring includes model fitting/prediction/traversal; timing categories overlap.\',\n                            \'Emit includes locking, score verification, serialization and queue backpressure.\',\n                            \'Upload time overlaps computation; final drain alone is not total checkpoint overhead.\',\n                            \'Profiling adds overhead; three flow cases are not a full-universe runtime estimate.\',\n                            \'Predictions independently traversed; saved scores compared; no new reference retraining.\',\n                            \'Interrupted profile may restart at most 21 fits. Completed matching receipt is reused.\',\n                            \'Full screen remains disabled; no automatic pilot identity migration.\']\n    assert len(output[\'fits\']) == 21 and len(uploads) == 21\n    output[\'status\'] = \'PASS_PROFILE_AND_SAVED_SCORES\'\n    # No other HTTP activity on writer_store remains; final receipt write is untimed.\n    writer_store.json(\'RUNTIME_PROFILE.json\', output)\n    print(output[\'status\'])\n    print(json.dumps(output[\'summary\'], indent=2))\n    print(\'Saved RUNTIME_PROFILE.json. Send this summary; no full screen started.\')\n', 'worker': '"""Compare execution backends only; no experiment settings are changed."""\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\nimport threading\n\n\ndef worker_case(sources, task, label=\'case\'):\n    # Executable source is packaged with hashes in the notebook. Each worker gets\n    # arrays and pure model code only: no Drive clients, credentials or file writes.\n    import numpy as np\n    ref = {}\n    exec(compile(sources[\'reference\'], \'screen_runner.py\', \'exec\'), ref)\n    opt = {\'REF\':ref}\n    exec(compile(sources[\'optimized\'], \'screen_throughput.py\', \'exec\'), opt)\n    start, cpu = time.perf_counter(), time.thread_time()\n    print(label+\' started\', flush=True)\n    def emit(key, score):\n        print(label+\' \'+key+\' complete\', flush=True)\n    result = opt[\'fold_task\'](*task, True, emit)\n    compact = {\'selected\':result[\'selected\'],\n               \'scores\':{k:opt[\'scientific\'](v) for k,v in result[\'fits\'].items()},\n               \'predictions\':{}}\n    for key, values in result[\'predictions\'].items():\n        if values is None:\n            compact[\'predictions\'][key] = None\n        else:\n            values = np.ascontiguousarray(values, dtype=\'<f8\')\n            compact[\'predictions\'][key] = {\'rows\':len(values), \'sha256\':hashlib.sha256(values.tobytes()).hexdigest()}\n    compact[\'timing\'] = {\'task_wall_seconds\':time.perf_counter()-start,\n                         \'task_thread_cpu_seconds\':time.thread_time()-cpu}\n    return compact\n\n\ndef scientific_case(case):\n    return {k:v for k,v in case.items() if k != \'timing\'}\n\n\ndef run_backend(sources, tasks, backend, workers, timeout_seconds=240):\n    from joblib import Parallel, delayed, parallel_config\n    from threadpoolctl import threadpool_limits\n    # Initialize tree imports in the parent before starting threads or setting limits.\n    from sklearn.tree import DecisionTreeRegressor\n    assert backend in (\'threading\',\'loky\') and workers in (1,2)\n    start = time.perf_counter()\n    # Disable automatic memmapping: no dataset/prediction files on worker disks.\n    options = dict(backend=backend, n_jobs=workers, max_nbytes=None)\n    if backend == \'loky\':\n        options[\'inner_max_num_threads\'] = 1\n    print(f\'Starting {backend}: {workers} workers; {timeout_seconds}s task timeout\',flush=True)\n    done = threading.Event()\n    def heartbeat():\n        while not done.wait(20):\n            print(f\'{backend}: still waiting; {time.perf_counter()-start:.0f}s elapsed\',flush=True)\n    monitor = threading.Thread(target=heartbeat,daemon=True)\n    monitor.start()\n    try:\n        with threadpool_limits(limits=1), parallel_config(**options):\n            result = Parallel(batch_size=1, pre_dispatch=workers, timeout=timeout_seconds)(\n                delayed(worker_case)(sources, task, f\'{backend} case {i+1}/{len(tasks)}\')\n                for i,task in enumerate(tasks))\n    except BaseException:\n        print(\'Comparison incomplete. Restart the Colab session before retrying: interrupted thread fits may still be running. No result is approved.\',flush=True)\n        raise\n    finally:\n        done.set()\n        monitor.join(timeout=1)\n    return {\'wall_seconds_including_dispatch\':time.perf_counter()-start, \'cases\':result}\n\n\ndef run_check(config, expected_identity, sources, hashes):\n    import numpy as np\n    import pandas as pd\n    import sklearn\n    import pyarrow\n    import joblib\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    assert all(hashlib.sha256(source.encode()).hexdigest()==hashes[k] for k,source in sources.items())\n    auth.authenticate_user()\n    print(\'Authentication complete; checking saved identities and receipts.\',flush=True)\n    credentials,_ = google.auth.default()\n    service = build(\'drive\',\'v3\',credentials=credentials,cache_discovery=False)\n    store = REF[\'DriveStore\'](service,config[\'folder\'])\n    saved = json.loads(store.read(\'SCREEN_RESULTS.json\'))\n    versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,\n                    sklearn=sklearn.__version__, pyarrow=pyarrow.__version__)\n    assert saved[\'identity\']==expected_identity and REF[\'signature\'](expected_identity)==saved[\'signature\']\n    assert expected_identity[\'config\']==config and expected_identity[\'versions\']==versions, \'Runtime/config changed\'\n    raw = store.read(\'RUNTIME_PROFILE.json\')\n    assert raw, \'Passing runtime profile required\'\n    profile = json.loads(raw)\n    assert profile[\'status\']==\'PASS_PROFILE_AND_SAVED_SCORES\'\n    assert REF[\'signature\'](profile[\'identity\'])==profile[\'signature\']\n    assert profile[\'identity\'][\'pilot_signature\']==saved[\'signature\']\n    assert profile[\'identity\'][\'hashes\']=={k:hashes[k] for k in (\'reference\',\'optimized\',\'profile\')}\n    for k,file_id in config[\'source_ids\'].items():\n        assert service.files().get(fileId=file_id,fields=\'id,name,size,md5Checksum,modifiedTime\').execute()==config[\'audit_sources\'][k]\n    workers = min(2, joblib.cpu_count())\n    identity = {\'hashes\':hashes,\'pilot_signature\':saved[\'signature\'],\'profile_sha256\':REF[\'sha\'](raw),\n                \'versions\':versions,\'joblib\':joblib.__version__,\'workers\':workers,\n                \'scope\':\'three fixed pairs, fold 5; threading then loky; 42 fits\'}\n    sig = REF[\'signature\'](identity)\n    prior = store.read(\'WORKER_CHECK.json\')\n    if prior:\n        prior = json.loads(prior)\n        if prior.get(\'signature\')==sig and prior.get(\'status\')==\'PASS_WORKER_EQUIVALENCE\':\n            print(\'Completed check reused; no fits repeated.\')\n            print(json.dumps(prior[\'summary\'],indent=2))\n            return\n    if workers < 2:\n        print(\'Only one usable CPU detected. Parallel-worker comparison skipped; no fits run.\')\n        return\n    print(\'Identity checks passed; loading and validating existing caches.\',flush=True)\n    tasks, expected, cases = PROFILE[\'load_tasks\'](store,service,config,saved)\n    print(\'Caches ready. Starting worker comparison.\',flush=True)\n    output = {\'identity\':identity,\'signature\':sig,\'status\':\'RUNNING_WORKER_CHECK\',\n              \'cases\':cases,\'phases\':{},\'full_screen_authorized\':False}\n    for backend in (\'threading\',\'loky\'):\n        phase = run_backend(sources,tasks,backend,workers)\n        for i,result in enumerate(phase[\'cases\']):\n            assert result[\'scores\']=={k:OPT[\'scientific\'](v) for k,v in expected[i].items()}, \'Saved score mismatch\'\n            selected = REF[\'pick_setting\']([v for k,v in expected[i].items() if k.startswith(\'inner|\')])\n            assert result[\'selected\']==(None if selected is None else [selected[\'depth\'],selected[\'min_samples_leaf\']])\n        output[\'phases\'][backend] = phase\n        # No uploads inside either timed phase; identical checkpoint policy for both.\n        store.json(\'WORKER_CHECK.json\',output)\n        print(backend+\' complete: \'+str(round(phase[\'wall_seconds_including_dispatch\'],2))+\' seconds\',flush=True)\n    a,b = output[\'phases\'][\'threading\'],output[\'phases\'][\'loky\']\n    assert [scientific_case(v) for v in a[\'cases\']]==[scientific_case(v) for v in b[\'cases\']], \'Predictions, scores or selections differ\'\n    output[\'summary\'] = {\'fits\':42,\'workers\':workers,\n                         \'threads_seconds\':a[\'wall_seconds_including_dispatch\'],\n                         \'processes_seconds\':b[\'wall_seconds_including_dispatch\'],\n                         \'process_speedup_ratio\':a[\'wall_seconds_including_dispatch\']/b[\'wall_seconds_including_dispatch\'],\n                         \'exact_prediction_digest_match\':True,\'saved_scores_and_selections_match\':True,\n                         \'automatic_promotion\':False}\n    output[\'limitations\'] = [\'One fixed-order comparison on three flow cases; order/cache/noise can affect timing.\',\n                            \'Includes dispatch, worker startup and serialization; checkpoint uploads excluded equally.\',\n                            \'Bounded to two workers and one native thread per worker. No automatic worker-count expansion.\',\n                            \'Prediction digests cover every evaluated row for all inner and outer fits; raw predictions stay in memory.\',\n                            \'Not full-universe equivalence, memory-capacity certification or a full-run runtime estimate.\',\n                            \'Interrupted comparison restarts at most 42 diagnostic fits; matching PASS receipt is reused.\',\n                            \'No adoption or full screen without review.\']\n    output[\'status\']=\'PASS_WORKER_EQUIVALENCE\'\n    store.json(\'WORKER_CHECK.json\',output)\n    print(output[\'status\'])\n    print(json.dumps(output[\'summary\'],indent=2))\n'}
hashes = {'reference': 'e584210af02adb15168245cd13e2cb6603eed882069cb75cd877d29122a6821b', 'optimized': '29fd33d96055bc3fe20829806211d34333ec4e0be12fb676c9bb37d4d0f5ed7b', 'profile': '08a47dd56271ca5943ada2c6e0c31cc49309183bac21fa6d6e5b2a4180fc6774', 'worker': '5c7e700cfe2dabb591d19f5c1b9b8017172365c78040b8ebde7035ea27de8eef'}
assert all(hashlib.sha256(v.encode()).hexdigest()==hashes[k] for k,v in sources.items())
REF = {}
exec(compile(sources["reference"], "screen_runner.py", "exec"), REF)
OPT = {"REF":REF}
exec(compile(sources["optimized"], "screen_throughput.py", "exec"), OPT)
PROFILE = {"REF":REF,"OPT":OPT}
exec(compile(sources["profile"], "screen_runtime_profile.py", "exec"), PROFILE)
WORKER = {"REF":REF,"OPT":OPT,"PROFILE":PROFILE}
exec(compile(sources["worker"], "screen_worker_check.py", "exec"), WORKER)
config = json.loads('{"version": "btc_time_high_single_feature_screen_v1", "symbol": "BTCUSDT", "target": "time_to_high_min", "folder": "1DXhoOLybfhEIt3phvPfG3cL3M31FPFEC", "source_ids": {"X": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "Y": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM"}, "audit_sources": {"X": {"id": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "name": "Handoff 1_F4_f4Z.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "043b2f570cfb63215749385d72c7fe59", "size": "15600742757"}, "Y": {"id": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM", "name": "HANDOFF_2_CANONICAL_XY_V1_1.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "bf2cb8dd274222747fde1b2e979b42b7", "size": "3033662926"}}, "audit_receipt_sha256": "6daca0c3d14b09de5695cd83cc3e86418f39d8afff1f4116611c0ed7cf5d6937", "baseline_signature": "729209d72e1ee50796f4964f087eb66b7e4fb87a6200e926cf8ef838af92272e", "candidate_features": ["F4Z__basis__basis_change_percentile__12h__value", "F4Z__basis__basis_change_percentile__15m__value", "F4Z__basis__basis_change_percentile__1h__value", "F4Z__basis__basis_change_percentile__4h__value", "F4Z__basis__basis_change_percentile__5m__value", "F4Z__basis__basis_change_rz__12h__value", "F4Z__basis__basis_change_rz__15m__value", "F4Z__basis__basis_change_rz__1h__value", "F4Z__basis__basis_change_rz__4h__value", "F4Z__basis__basis_change_rz__5m__value", "F4Z__basis__basis_level_percentile__12h__value", "F4Z__basis__basis_level_percentile__15m__value", "F4Z__basis__basis_level_percentile__1h__value", "F4Z__basis__basis_level_percentile__4h__value", "F4Z__basis__basis_level_percentile__5m__value", "F4Z__basis__basis_level_rz__12h__value", "F4Z__basis__basis_level_rz__15m__value", "F4Z__basis__basis_level_rz__1h__value", "F4Z__basis__basis_level_rz__4h__value", "F4Z__basis__basis_level_rz__5m__value", "F4Z__basis__basis_path_retention_percentile__12h__value", "F4Z__basis__basis_path_retention_percentile__15m__value", "F4Z__basis__basis_path_retention_percentile__1h__value", "F4Z__basis__basis_path_retention_percentile__4h__value", "F4Z__basis__basis_path_retention_percentile__5m__value", "F4Z__basis__basis_persistence_percentile__12h__value", "F4Z__basis__basis_persistence_percentile__15m__value", "F4Z__basis__basis_persistence_percentile__1h__value", "F4Z__basis__basis_persistence_percentile__4h__value", "F4Z__basis__basis_persistence_percentile__5m__value", "F4Z__basis__basis_range_percentile__12h__value", "F4Z__basis__basis_range_percentile__15m__value", "F4Z__basis__basis_range_percentile__1h__value", "F4Z__basis__basis_range_percentile__4h__value", "F4Z__basis__basis_range_percentile__5m__value", "F4Z__basis__basis_range_rz__12h__value", "F4Z__basis__basis_range_rz__15m__value", "F4Z__basis__basis_range_rz__1h__value", "F4Z__basis__basis_range_rz__4h__value", "F4Z__basis__basis_range_rz__5m__value", "F4Z__flow__cvd_path_efficiency_percentile__12h__value", "F4Z__flow__cvd_path_efficiency_percentile__15m__value", "F4Z__flow__cvd_path_efficiency_percentile__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value", "F4Z__flow__cvd_path_efficiency_percentile__5m__value", "F4Z__flow__flow_acceleration_percentile__15m__12h__value", "F4Z__flow__flow_acceleration_percentile__15m__1h__value", "F4Z__flow__flow_acceleration_percentile__15m__4h__value", "F4Z__flow__flow_acceleration_percentile__1h__12h__value", "F4Z__flow__flow_acceleration_percentile__1h__4h__value", "F4Z__flow__flow_acceleration_percentile__4h__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__15m__value", "F4Z__flow__flow_acceleration_percentile__5m__1h__value", "F4Z__flow__flow_acceleration_percentile__5m__4h__value", "F4Z__flow__flow_acceleration_rz__15m__12h__value", "F4Z__flow__flow_acceleration_rz__15m__1h__value", "F4Z__flow__flow_acceleration_rz__15m__4h__value", "F4Z__flow__flow_acceleration_rz__1h__12h__value", "F4Z__flow__flow_acceleration_rz__1h__4h__value", "F4Z__flow__flow_acceleration_rz__4h__12h__value", "F4Z__flow__flow_acceleration_rz__5m__12h__value", "F4Z__flow__flow_acceleration_rz__5m__15m__value", "F4Z__flow__flow_acceleration_rz__5m__1h__value", "F4Z__flow__flow_acceleration_rz__5m__4h__value", "F4Z__flow__flow_activity_percentile__12h__value", "F4Z__flow__flow_activity_percentile__15m__value", "F4Z__flow__flow_activity_percentile__1h__value", "F4Z__flow__flow_activity_percentile__4h__value", "F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_activity_rz__12h__value", "F4Z__flow__flow_activity_rz__15m__value", "F4Z__flow__flow_activity_rz__1h__value", "F4Z__flow__flow_activity_rz__4h__value", "F4Z__flow__flow_activity_rz__5m__value", "F4Z__flow__flow_confirmation_percentile__12h__value", "F4Z__flow__flow_confirmation_percentile__15m__value", "F4Z__flow__flow_confirmation_percentile__1h__value", "F4Z__flow__flow_confirmation_percentile__4h__value", "F4Z__flow__flow_confirmation_percentile__5m__value", "F4Z__flow__flow_confirmation_rz__12h__value", "F4Z__flow__flow_confirmation_rz__15m__value", "F4Z__flow__flow_confirmation_rz__1h__value", "F4Z__flow__flow_confirmation_rz__4h__value", "F4Z__flow__flow_confirmation_rz__5m__value", "F4Z__flow__flow_cvd_slope_percentile__12h__value", "F4Z__flow__flow_cvd_slope_percentile__15m__value", "F4Z__flow__flow_cvd_slope_percentile__1h__value", "F4Z__flow__flow_cvd_slope_percentile__4h__value", "F4Z__flow__flow_cvd_slope_percentile__5m__value", "F4Z__flow__flow_cvd_slope_rz__12h__value", "F4Z__flow__flow_cvd_slope_rz__15m__value", "F4Z__flow__flow_cvd_slope_rz__1h__value", "F4Z__flow__flow_cvd_slope_rz__4h__value", "F4Z__flow__flow_cvd_slope_rz__5m__value", "F4Z__flow__flow_pressure_percentile__12h__value", "F4Z__flow__flow_pressure_percentile__15m__value", "F4Z__flow__flow_pressure_percentile__1h__value", "F4Z__flow__flow_pressure_percentile__4h__value", "F4Z__flow__flow_pressure_percentile__5m__value", "F4Z__flow__flow_pressure_rz__12h__value", "F4Z__flow__flow_pressure_rz__15m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__flow_pressure_rz__4h__value", "F4Z__flow__flow_pressure_rz__5m__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__4h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__15m__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__4h__value", "F4Z__flow__signed_flow_acceleration_rz__4h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__15m__value", "F4Z__flow__signed_flow_acceleration_rz__5m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__4h__value", "F4Z__funding__funding_change_percentile__15m__value", "F4Z__funding__funding_change_percentile__1h__value", "F4Z__funding__funding_change_percentile__4h__value", "F4Z__funding__funding_change_percentile__5m__value", "F4Z__funding__funding_change_rz__15m__value", "F4Z__funding__funding_change_rz__1h__value", "F4Z__funding__funding_change_rz__4h__value", "F4Z__funding__funding_change_rz__5m__value", "F4Z__funding__funding_level_percentile__15m__value", "F4Z__funding__funding_level_percentile__1h__value", "F4Z__funding__funding_level_percentile__4h__value", "F4Z__funding__funding_level_percentile__5m__value", "F4Z__funding__funding_level_rz__15m__value", "F4Z__funding__funding_level_rz__1h__value", "F4Z__funding__funding_level_rz__4h__value", "F4Z__funding__funding_level_rz__5m__value", "F4Z__funding__funding_path_retention_percentile__15m__value", "F4Z__funding__funding_path_retention_percentile__1h__value", "F4Z__funding__funding_path_retention_percentile__4h__value", "F4Z__funding__funding_path_retention_percentile__5m__value", "F4Z__funding__funding_persistence_percentile__15m__value", "F4Z__funding__funding_persistence_percentile__1h__value", "F4Z__funding__funding_persistence_percentile__4h__value", "F4Z__funding__funding_persistence_percentile__5m__value", "F4Z__funding__funding_range_percentile__15m__value", "F4Z__funding__funding_range_percentile__1h__value", "F4Z__funding__funding_range_percentile__4h__value", "F4Z__funding__funding_range_percentile__5m__value", "F4Z__funding__funding_range_rz__15m__value", "F4Z__funding__funding_range_rz__1h__value", "F4Z__funding__funding_range_rz__4h__value", "F4Z__funding__funding_range_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__5m__value", "F4Z__liquidation__liq_acceleration_percentile__15m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__4h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__4h__value", "F4Z__liquidation__liq_acceleration_percentile__4h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__15m__value", "F4Z__liquidation__liq_acceleration_percentile__5m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__4h__value", "F4Z__liquidation__liq_acceleration_rz__15m__12h__value", "F4Z__liquidation__liq_acceleration_rz__15m__1h__value", "F4Z__liquidation__liq_acceleration_rz__15m__4h__value", "F4Z__liquidation__liq_acceleration_rz__1h__12h__value", "F4Z__liquidation__liq_acceleration_rz__1h__4h__value", "F4Z__liquidation__liq_acceleration_rz__4h__12h__value", "F4Z__liquidation__liq_acceleration_rz__5m__15m__value", "F4Z__liquidation__liq_acceleration_rz__5m__1h__value", "F4Z__liquidation__liq_acceleration_rz__5m__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__5m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__5m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__15m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__1h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__4h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__5m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__12h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__15m__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__1h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_percentile__12h__value", "F4Z__liquidation__liq_side_pressure_percentile__15m__value", "F4Z__liquidation__liq_side_pressure_percentile__1h__value", "F4Z__liquidation__liq_side_pressure_percentile__4h__value", "F4Z__liquidation__liq_side_pressure_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_rz__12h__value", "F4Z__liquidation__liq_side_pressure_rz__15m__value", "F4Z__liquidation__liq_side_pressure_rz__1h__value", "F4Z__liquidation__liq_side_pressure_rz__4h__value", "F4Z__liquidation__liq_side_pressure_rz__5m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__12h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__15m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__1h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__4h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__5m__value", "F4Z__liquidation__venue_breadth_percentile__12h__value", "F4Z__liquidation__venue_breadth_percentile__15m__value", "F4Z__liquidation__venue_breadth_percentile__1h__value", "F4Z__liquidation__venue_breadth_percentile__4h__value", "F4Z__liquidation__venue_breadth_percentile__5m__value", "F4Z__liquidation__venue_fragmentation_percentile__12h__value", "F4Z__liquidation__venue_fragmentation_percentile__15m__value", "F4Z__liquidation__venue_fragmentation_percentile__1h__value", "F4Z__liquidation__venue_fragmentation_percentile__4h__value", "F4Z__liquidation__venue_fragmentation_percentile__5m__value", "F4Z__oi__leverage_unwind_percentile__12h__value", "F4Z__oi__leverage_unwind_percentile__15m__value", "F4Z__oi__leverage_unwind_percentile__1h__value", "F4Z__oi__leverage_unwind_percentile__4h__value", "F4Z__oi__leverage_unwind_percentile__5m__value", "F4Z__oi__leverage_unwind_rz__12h__value", "F4Z__oi__leverage_unwind_rz__15m__value", "F4Z__oi__leverage_unwind_rz__1h__value", "F4Z__oi__leverage_unwind_rz__4h__value", "F4Z__oi__leverage_unwind_rz__5m__value", "F4Z__oi__oi_extreme_recency_percentile__12h__value", "F4Z__oi__oi_extreme_recency_percentile__15m__value", "F4Z__oi__oi_extreme_recency_percentile__1h__value", "F4Z__oi__oi_extreme_recency_percentile__4h__value", "F4Z__oi__oi_extreme_recency_percentile__5m__value", "F4Z__oi__oi_path_retention_percentile__12h__value", "F4Z__oi__oi_path_retention_percentile__15m__value", "F4Z__oi__oi_path_retention_percentile__1h__value", "F4Z__oi__oi_path_retention_percentile__4h__value", "F4Z__oi__oi_path_retention_percentile__5m__value", "F4Z__oi__oi_range_position_percentile__12h__value", "F4Z__oi__oi_range_position_percentile__15m__value", "F4Z__oi__oi_range_position_percentile__1h__value", "F4Z__oi__oi_range_position_percentile__4h__value", "F4Z__oi__oi_range_position_percentile__5m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__12h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__15m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__1h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__4h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__5m__value", "F4Z__price__price_acceleration_percentile__15m__12h__value", "F4Z__price__price_acceleration_percentile__15m__1h__value", "F4Z__price__price_acceleration_percentile__15m__4h__value", "F4Z__price__price_acceleration_percentile__1h__12h__value", "F4Z__price__price_acceleration_percentile__1h__4h__value", "F4Z__price__price_acceleration_percentile__4h__12h__value", "F4Z__price__price_acceleration_percentile__5m__12h__value", "F4Z__price__price_acceleration_percentile__5m__15m__value", "F4Z__price__price_acceleration_percentile__5m__1h__value", "F4Z__price__price_acceleration_percentile__5m__4h__value", "F4Z__price__price_acceleration_rz__15m__12h__value", "F4Z__price__price_acceleration_rz__15m__1h__value", "F4Z__price__price_acceleration_rz__15m__4h__value", "F4Z__price__price_acceleration_rz__1h__12h__value", "F4Z__price__price_acceleration_rz__1h__4h__value", "F4Z__price__price_acceleration_rz__4h__12h__value", "F4Z__price__price_acceleration_rz__5m__12h__value", "F4Z__price__price_acceleration_rz__5m__15m__value", "F4Z__price__price_acceleration_rz__5m__1h__value", "F4Z__price__price_acceleration_rz__5m__4h__value", "F4Z__price__price_displacement_percentile__12h__value", "F4Z__price__price_displacement_percentile__15m__value", "F4Z__price__price_displacement_percentile__1h__value", "F4Z__price__price_displacement_percentile__4h__value", "F4Z__price__price_displacement_percentile__5m__value", "F4Z__price__price_displacement_rz__12h__value", "F4Z__price__price_displacement_rz__15m__value", "F4Z__price__price_displacement_rz__1h__value", "F4Z__price__price_displacement_rz__4h__value", "F4Z__price__price_displacement_rz__5m__value", "F4Z__price__price_excursion_range_percentile__12h__value", "F4Z__price__price_excursion_range_percentile__15m__value", "F4Z__price__price_excursion_range_percentile__1h__value", "F4Z__price__price_excursion_range_percentile__4h__value", "F4Z__price__price_excursion_range_percentile__5m__value", "F4Z__price__price_excursion_range_rz__12h__value", "F4Z__price__price_excursion_range_rz__15m__value", "F4Z__price__price_excursion_range_rz__1h__value", "F4Z__price__price_excursion_range_rz__4h__value", "F4Z__price__price_excursion_range_rz__5m__value", "F4Z__price__price_move_retention_percentile__12h__value", "F4Z__price__price_move_retention_percentile__15m__value", "F4Z__price__price_move_retention_percentile__1h__value", "F4Z__price__price_move_retention_percentile__4h__value", "F4Z__price__price_move_retention_percentile__5m__value", "F4Z__volatility__range_expansion_percentile__15m__12h__value", "F4Z__volatility__range_expansion_percentile__15m__1h__value", "F4Z__volatility__range_expansion_percentile__15m__4h__value", "F4Z__volatility__range_expansion_percentile__1h__12h__value", "F4Z__volatility__range_expansion_percentile__1h__4h__value", "F4Z__volatility__range_expansion_percentile__4h__12h__value", "F4Z__volatility__range_expansion_percentile__5m__12h__value", "F4Z__volatility__range_expansion_percentile__5m__15m__value", "F4Z__volatility__range_expansion_percentile__5m__1h__value", "F4Z__volatility__range_expansion_percentile__5m__4h__value", "F4Z__volatility__range_expansion_rz__15m__12h__value", "F4Z__volatility__range_expansion_rz__15m__1h__value", "F4Z__volatility__range_expansion_rz__15m__4h__value", "F4Z__volatility__range_expansion_rz__1h__12h__value", "F4Z__volatility__range_expansion_rz__1h__4h__value", "F4Z__volatility__range_expansion_rz__4h__12h__value", "F4Z__volatility__range_expansion_rz__5m__12h__value", "F4Z__volatility__range_expansion_rz__5m__15m__value", "F4Z__volatility__range_expansion_rz__5m__1h__value", "F4Z__volatility__range_expansion_rz__5m__4h__value", "F4Z__volatility__realized_variance_level_percentile__12h__value", "F4Z__volatility__realized_variance_level_percentile__15m__value", "F4Z__volatility__realized_variance_level_percentile__1h__value", "F4Z__volatility__realized_variance_level_percentile__4h__value", "F4Z__volatility__realized_variance_level_percentile__5m__value", "F4Z__volatility__realized_variance_level_rz__12h__value", "F4Z__volatility__realized_variance_level_rz__15m__value", "F4Z__volatility__realized_variance_level_rz__1h__value", "F4Z__volatility__realized_variance_level_rz__4h__value", "F4Z__volatility__realized_variance_level_rz__5m__value", "F4Z__volatility__vol_acceleration_percentile__15m__12h__value", "F4Z__volatility__vol_acceleration_percentile__15m__1h__value", "F4Z__volatility__vol_acceleration_percentile__15m__4h__value", "F4Z__volatility__vol_acceleration_percentile__1h__12h__value", "F4Z__volatility__vol_acceleration_percentile__1h__4h__value", "F4Z__volatility__vol_acceleration_percentile__4h__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__15m__value", "F4Z__volatility__vol_acceleration_percentile__5m__1h__value", "F4Z__volatility__vol_acceleration_percentile__5m__4h__value", "F4Z__volatility__vol_acceleration_rz__15m__12h__value", "F4Z__volatility__vol_acceleration_rz__15m__1h__value", "F4Z__volatility__vol_acceleration_rz__15m__4h__value", "F4Z__volatility__vol_acceleration_rz__1h__12h__value", "F4Z__volatility__vol_acceleration_rz__1h__4h__value", "F4Z__volatility__vol_acceleration_rz__4h__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__15m__value", "F4Z__volatility__vol_acceleration_rz__5m__1h__value", "F4Z__volatility__vol_acceleration_rz__5m__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__12h__value", "F4Z__volatility__vol_shock_concentration_percentile__15m__value", "F4Z__volatility__vol_shock_concentration_percentile__1h__value", "F4Z__volatility__vol_shock_concentration_percentile__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__5m__value", "F4Z__volatility__vol_shock_recency_percentile__12h__value", "F4Z__volatility__vol_shock_recency_percentile__15m__value", "F4Z__volatility__vol_shock_recency_percentile__1h__value", "F4Z__volatility__vol_shock_recency_percentile__4h__value", "F4Z__volatility__vol_shock_recency_percentile__5m__value"], "pilot_features": ["F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value"], "pilot_horizons": [15, 2880, 18000], "horizons": [15, 60, 240, 480, 960, 2880, 4320, 7200, 12000, 18000], "months": ["2022-04", "2022-05", "2022-06", "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12", "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06", "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"], "origin_end_exclusive": "2024-07-01T00:00:00Z", "reserve_start": "2024-07-24T00:01:00Z", "cadence_minutes": 30, "folds": [["2023-04-01", "2023-07-01"], ["2023-07-01", "2023-10-01"], ["2023-10-01", "2024-01-01"], ["2024-01-01", "2024-04-01"], ["2024-04-01", "2024-07-01"]], "grid": [[2, 64], [2, 256], [3, 64], [3, 256], [5, 64], [5, 256]], "minimum_observed_training": 1000, "inner_validation": "Preceding three calendar months; inner training label_end < inner start, inner validation label_end < outer start. Outer training label_end < outer start.", "model": "Single raw F4Z input plus explicit missing indicator; DecisionTreeRegressor absolute_error, seed48, depth/leaf grid only.", "preprocessing": "Train-only finite-value median imputation plus missing indicator. No feature scaling. Baseline_n >=1000, finite, valid, recorded available_after <= origin, scientific_status PROVISIONAL explicitly allowed for provisional research only.", "selection": "Lowest inner MAE; ties shallower depth then larger leaf. Never use outer MAE to choose fold settings.", "ineligible_policy": "If fewer than1000 observed training values or constant observed training feature: no fit; if no eligible inner setting or outer training insufficient, predict outer-training target median on all outer rows. Report fallback and coverage.", "evaluation": "Exact original baseline rows; primary MAE minutes, secondary percent of horizon; pooled row-weighted errors; compare all three baselines. Record missing/observed subset errors. No significance/global optimum/trading claim.", "pilot_scope": "3 flow features x3 horizons x5 folds x(6 inner+1 outer) = at most315 fits. Features selected for differing market meanings and sampled coverage, not outcome scores.", "pilot_meanings": {"F4Z__flow__flow_activity_percentile__5m__value": "Recent flow activity relative to history", "F4Z__flow__flow_pressure_rz__1h__value": "Directional trading pressure relative to history", "F4Z__flow__cvd_path_efficiency_percentile__4h__value": "How consistently cumulative trading flow follows one direction"}, "cache_policy": "One compact research-only NPZ per horizon/family in Drive, no PC cache. Float64 values preserve source conversion. Checkpoint each month; immutable signature/hash checks. Family cache includes all candidate columns from that family (85 flow) to avoid rereading later. <=128MiB per cache.", "checkpoint_policy": "Per-fit compact JSON scores in Timing_Screen_Fit_Scores; no model binaries or row predictions. Partial family/horizon data resume completed months. An interrupted unsaved fit may restart. Pilot completed fits reused by full mode under identical identity/versions.", "workers": "Up to4 bounded I/O workers; up to2 fit workers; native thread pools limited. No automatic full run after pilot.", "limitations": ["F4Z upstream timing/normalization remains unresolved; PROVISIONAL research only.", "Sample-based candidate preselection does not prove excluded features useless across full history.", "Single-feature trees do not establish globally best features or combinations.", "Reserved period previously exposed in other project work but excluded from this task.", "Errors overlap in time; no iid significance assumptions."], "full_launch_policy": "Review pilot correctness, runtime, memory and results before authorizing full386-feature expansion. Do not change to full mode yet."}')
pilot_identity = json.loads('{"config": {"version": "btc_time_high_single_feature_screen_v1", "symbol": "BTCUSDT", "target": "time_to_high_min", "folder": "1DXhoOLybfhEIt3phvPfG3cL3M31FPFEC", "source_ids": {"X": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "Y": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM"}, "audit_sources": {"X": {"id": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "name": "Handoff 1_F4_f4Z.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "043b2f570cfb63215749385d72c7fe59", "size": "15600742757"}, "Y": {"id": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM", "name": "HANDOFF_2_CANONICAL_XY_V1_1.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "bf2cb8dd274222747fde1b2e979b42b7", "size": "3033662926"}}, "audit_receipt_sha256": "6daca0c3d14b09de5695cd83cc3e86418f39d8afff1f4116611c0ed7cf5d6937", "baseline_signature": "729209d72e1ee50796f4964f087eb66b7e4fb87a6200e926cf8ef838af92272e", "candidate_features": ["F4Z__basis__basis_change_percentile__12h__value", "F4Z__basis__basis_change_percentile__15m__value", "F4Z__basis__basis_change_percentile__1h__value", "F4Z__basis__basis_change_percentile__4h__value", "F4Z__basis__basis_change_percentile__5m__value", "F4Z__basis__basis_change_rz__12h__value", "F4Z__basis__basis_change_rz__15m__value", "F4Z__basis__basis_change_rz__1h__value", "F4Z__basis__basis_change_rz__4h__value", "F4Z__basis__basis_change_rz__5m__value", "F4Z__basis__basis_level_percentile__12h__value", "F4Z__basis__basis_level_percentile__15m__value", "F4Z__basis__basis_level_percentile__1h__value", "F4Z__basis__basis_level_percentile__4h__value", "F4Z__basis__basis_level_percentile__5m__value", "F4Z__basis__basis_level_rz__12h__value", "F4Z__basis__basis_level_rz__15m__value", "F4Z__basis__basis_level_rz__1h__value", "F4Z__basis__basis_level_rz__4h__value", "F4Z__basis__basis_level_rz__5m__value", "F4Z__basis__basis_path_retention_percentile__12h__value", "F4Z__basis__basis_path_retention_percentile__15m__value", "F4Z__basis__basis_path_retention_percentile__1h__value", "F4Z__basis__basis_path_retention_percentile__4h__value", "F4Z__basis__basis_path_retention_percentile__5m__value", "F4Z__basis__basis_persistence_percentile__12h__value", "F4Z__basis__basis_persistence_percentile__15m__value", "F4Z__basis__basis_persistence_percentile__1h__value", "F4Z__basis__basis_persistence_percentile__4h__value", "F4Z__basis__basis_persistence_percentile__5m__value", "F4Z__basis__basis_range_percentile__12h__value", "F4Z__basis__basis_range_percentile__15m__value", "F4Z__basis__basis_range_percentile__1h__value", "F4Z__basis__basis_range_percentile__4h__value", "F4Z__basis__basis_range_percentile__5m__value", "F4Z__basis__basis_range_rz__12h__value", "F4Z__basis__basis_range_rz__15m__value", "F4Z__basis__basis_range_rz__1h__value", "F4Z__basis__basis_range_rz__4h__value", "F4Z__basis__basis_range_rz__5m__value", "F4Z__flow__cvd_path_efficiency_percentile__12h__value", "F4Z__flow__cvd_path_efficiency_percentile__15m__value", "F4Z__flow__cvd_path_efficiency_percentile__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value", "F4Z__flow__cvd_path_efficiency_percentile__5m__value", "F4Z__flow__flow_acceleration_percentile__15m__12h__value", "F4Z__flow__flow_acceleration_percentile__15m__1h__value", "F4Z__flow__flow_acceleration_percentile__15m__4h__value", "F4Z__flow__flow_acceleration_percentile__1h__12h__value", "F4Z__flow__flow_acceleration_percentile__1h__4h__value", "F4Z__flow__flow_acceleration_percentile__4h__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__12h__value", "F4Z__flow__flow_acceleration_percentile__5m__15m__value", "F4Z__flow__flow_acceleration_percentile__5m__1h__value", "F4Z__flow__flow_acceleration_percentile__5m__4h__value", "F4Z__flow__flow_acceleration_rz__15m__12h__value", "F4Z__flow__flow_acceleration_rz__15m__1h__value", "F4Z__flow__flow_acceleration_rz__15m__4h__value", "F4Z__flow__flow_acceleration_rz__1h__12h__value", "F4Z__flow__flow_acceleration_rz__1h__4h__value", "F4Z__flow__flow_acceleration_rz__4h__12h__value", "F4Z__flow__flow_acceleration_rz__5m__12h__value", "F4Z__flow__flow_acceleration_rz__5m__15m__value", "F4Z__flow__flow_acceleration_rz__5m__1h__value", "F4Z__flow__flow_acceleration_rz__5m__4h__value", "F4Z__flow__flow_activity_percentile__12h__value", "F4Z__flow__flow_activity_percentile__15m__value", "F4Z__flow__flow_activity_percentile__1h__value", "F4Z__flow__flow_activity_percentile__4h__value", "F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_activity_rz__12h__value", "F4Z__flow__flow_activity_rz__15m__value", "F4Z__flow__flow_activity_rz__1h__value", "F4Z__flow__flow_activity_rz__4h__value", "F4Z__flow__flow_activity_rz__5m__value", "F4Z__flow__flow_confirmation_percentile__12h__value", "F4Z__flow__flow_confirmation_percentile__15m__value", "F4Z__flow__flow_confirmation_percentile__1h__value", "F4Z__flow__flow_confirmation_percentile__4h__value", "F4Z__flow__flow_confirmation_percentile__5m__value", "F4Z__flow__flow_confirmation_rz__12h__value", "F4Z__flow__flow_confirmation_rz__15m__value", "F4Z__flow__flow_confirmation_rz__1h__value", "F4Z__flow__flow_confirmation_rz__4h__value", "F4Z__flow__flow_confirmation_rz__5m__value", "F4Z__flow__flow_cvd_slope_percentile__12h__value", "F4Z__flow__flow_cvd_slope_percentile__15m__value", "F4Z__flow__flow_cvd_slope_percentile__1h__value", "F4Z__flow__flow_cvd_slope_percentile__4h__value", "F4Z__flow__flow_cvd_slope_percentile__5m__value", "F4Z__flow__flow_cvd_slope_rz__12h__value", "F4Z__flow__flow_cvd_slope_rz__15m__value", "F4Z__flow__flow_cvd_slope_rz__1h__value", "F4Z__flow__flow_cvd_slope_rz__4h__value", "F4Z__flow__flow_cvd_slope_rz__5m__value", "F4Z__flow__flow_pressure_percentile__12h__value", "F4Z__flow__flow_pressure_percentile__15m__value", "F4Z__flow__flow_pressure_percentile__1h__value", "F4Z__flow__flow_pressure_percentile__4h__value", "F4Z__flow__flow_pressure_percentile__5m__value", "F4Z__flow__flow_pressure_rz__12h__value", "F4Z__flow__flow_pressure_rz__15m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__flow_pressure_rz__4h__value", "F4Z__flow__flow_pressure_rz__5m__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__15m__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__1h__4h__value", "F4Z__flow__signed_flow_acceleration_percentile__4h__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__12h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__15m__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__1h__value", "F4Z__flow__signed_flow_acceleration_percentile__5m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__15m__4h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__1h__4h__value", "F4Z__flow__signed_flow_acceleration_rz__4h__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__12h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__15m__value", "F4Z__flow__signed_flow_acceleration_rz__5m__1h__value", "F4Z__flow__signed_flow_acceleration_rz__5m__4h__value", "F4Z__funding__funding_change_percentile__15m__value", "F4Z__funding__funding_change_percentile__1h__value", "F4Z__funding__funding_change_percentile__4h__value", "F4Z__funding__funding_change_percentile__5m__value", "F4Z__funding__funding_change_rz__15m__value", "F4Z__funding__funding_change_rz__1h__value", "F4Z__funding__funding_change_rz__4h__value", "F4Z__funding__funding_change_rz__5m__value", "F4Z__funding__funding_level_percentile__15m__value", "F4Z__funding__funding_level_percentile__1h__value", "F4Z__funding__funding_level_percentile__4h__value", "F4Z__funding__funding_level_percentile__5m__value", "F4Z__funding__funding_level_rz__15m__value", "F4Z__funding__funding_level_rz__1h__value", "F4Z__funding__funding_level_rz__4h__value", "F4Z__funding__funding_level_rz__5m__value", "F4Z__funding__funding_path_retention_percentile__15m__value", "F4Z__funding__funding_path_retention_percentile__1h__value", "F4Z__funding__funding_path_retention_percentile__4h__value", "F4Z__funding__funding_path_retention_percentile__5m__value", "F4Z__funding__funding_persistence_percentile__15m__value", "F4Z__funding__funding_persistence_percentile__1h__value", "F4Z__funding__funding_persistence_percentile__4h__value", "F4Z__funding__funding_persistence_percentile__5m__value", "F4Z__funding__funding_range_percentile__15m__value", "F4Z__funding__funding_range_percentile__1h__value", "F4Z__funding__funding_range_percentile__4h__value", "F4Z__funding__funding_range_percentile__5m__value", "F4Z__funding__funding_range_rz__15m__value", "F4Z__funding__funding_range_rz__1h__value", "F4Z__funding__funding_range_rz__4h__value", "F4Z__funding__funding_range_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_percentile__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__12h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__4h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_active_severity_rz__5m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__15m__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__1h__value", "F4Z__liquidation__cross_asset_liq_activity_raw_historical_active_rate__5m__value", "F4Z__liquidation__liq_acceleration_percentile__15m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__15m__4h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__1h__4h__value", "F4Z__liquidation__liq_acceleration_percentile__4h__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__12h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__15m__value", "F4Z__liquidation__liq_acceleration_percentile__5m__1h__value", "F4Z__liquidation__liq_acceleration_percentile__5m__4h__value", "F4Z__liquidation__liq_acceleration_rz__15m__12h__value", "F4Z__liquidation__liq_acceleration_rz__15m__1h__value", "F4Z__liquidation__liq_acceleration_rz__15m__4h__value", "F4Z__liquidation__liq_acceleration_rz__1h__12h__value", "F4Z__liquidation__liq_acceleration_rz__1h__4h__value", "F4Z__liquidation__liq_acceleration_rz__4h__12h__value", "F4Z__liquidation__liq_acceleration_rz__5m__15m__value", "F4Z__liquidation__liq_acceleration_rz__5m__1h__value", "F4Z__liquidation__liq_acceleration_rz__5m__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_percentile__5m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__12h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__15m__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__1h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__4h__value", "F4Z__liquidation__liq_activity_severity_active_severity_rz__5m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__15m__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__1h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__4h__value", "F4Z__liquidation__liq_activity_severity_historical_active_rate__5m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_percentile__5m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__15m__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__1h__4h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__4h__12h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__15m__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__1h__value", "F4Z__liquidation__liq_fast_slow_lift_rz__5m__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__12h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__15m__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__1h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__4h__value", "F4Z__liquidation__liq_peak_age_seconds_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_percentile__12h__value", "F4Z__liquidation__liq_side_pressure_percentile__15m__value", "F4Z__liquidation__liq_side_pressure_percentile__1h__value", "F4Z__liquidation__liq_side_pressure_percentile__4h__value", "F4Z__liquidation__liq_side_pressure_percentile__5m__value", "F4Z__liquidation__liq_side_pressure_rz__12h__value", "F4Z__liquidation__liq_side_pressure_rz__15m__value", "F4Z__liquidation__liq_side_pressure_rz__1h__value", "F4Z__liquidation__liq_side_pressure_rz__4h__value", "F4Z__liquidation__liq_side_pressure_rz__5m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__12h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__15m__value", "F4Z__liquidation__liq_temporal_concentration_percentile__1h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__4h__value", "F4Z__liquidation__liq_temporal_concentration_percentile__5m__value", "F4Z__liquidation__venue_breadth_percentile__12h__value", "F4Z__liquidation__venue_breadth_percentile__15m__value", "F4Z__liquidation__venue_breadth_percentile__1h__value", "F4Z__liquidation__venue_breadth_percentile__4h__value", "F4Z__liquidation__venue_breadth_percentile__5m__value", "F4Z__liquidation__venue_fragmentation_percentile__12h__value", "F4Z__liquidation__venue_fragmentation_percentile__15m__value", "F4Z__liquidation__venue_fragmentation_percentile__1h__value", "F4Z__liquidation__venue_fragmentation_percentile__4h__value", "F4Z__liquidation__venue_fragmentation_percentile__5m__value", "F4Z__oi__leverage_unwind_percentile__12h__value", "F4Z__oi__leverage_unwind_percentile__15m__value", "F4Z__oi__leverage_unwind_percentile__1h__value", "F4Z__oi__leverage_unwind_percentile__4h__value", "F4Z__oi__leverage_unwind_percentile__5m__value", "F4Z__oi__leverage_unwind_rz__12h__value", "F4Z__oi__leverage_unwind_rz__15m__value", "F4Z__oi__leverage_unwind_rz__1h__value", "F4Z__oi__leverage_unwind_rz__4h__value", "F4Z__oi__leverage_unwind_rz__5m__value", "F4Z__oi__oi_extreme_recency_percentile__12h__value", "F4Z__oi__oi_extreme_recency_percentile__15m__value", "F4Z__oi__oi_extreme_recency_percentile__1h__value", "F4Z__oi__oi_extreme_recency_percentile__4h__value", "F4Z__oi__oi_extreme_recency_percentile__5m__value", "F4Z__oi__oi_path_retention_percentile__12h__value", "F4Z__oi__oi_path_retention_percentile__15m__value", "F4Z__oi__oi_path_retention_percentile__1h__value", "F4Z__oi__oi_path_retention_percentile__4h__value", "F4Z__oi__oi_path_retention_percentile__5m__value", "F4Z__oi__oi_range_position_percentile__12h__value", "F4Z__oi__oi_range_position_percentile__15m__value", "F4Z__oi__oi_range_position_percentile__1h__value", "F4Z__oi__oi_range_position_percentile__4h__value", "F4Z__oi__oi_range_position_percentile__5m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__12h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__15m__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__1h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__4h__value", "F4Z__oi__remaining_recent_oi_fraction_percentile__5m__value", "F4Z__price__price_acceleration_percentile__15m__12h__value", "F4Z__price__price_acceleration_percentile__15m__1h__value", "F4Z__price__price_acceleration_percentile__15m__4h__value", "F4Z__price__price_acceleration_percentile__1h__12h__value", "F4Z__price__price_acceleration_percentile__1h__4h__value", "F4Z__price__price_acceleration_percentile__4h__12h__value", "F4Z__price__price_acceleration_percentile__5m__12h__value", "F4Z__price__price_acceleration_percentile__5m__15m__value", "F4Z__price__price_acceleration_percentile__5m__1h__value", "F4Z__price__price_acceleration_percentile__5m__4h__value", "F4Z__price__price_acceleration_rz__15m__12h__value", "F4Z__price__price_acceleration_rz__15m__1h__value", "F4Z__price__price_acceleration_rz__15m__4h__value", "F4Z__price__price_acceleration_rz__1h__12h__value", "F4Z__price__price_acceleration_rz__1h__4h__value", "F4Z__price__price_acceleration_rz__4h__12h__value", "F4Z__price__price_acceleration_rz__5m__12h__value", "F4Z__price__price_acceleration_rz__5m__15m__value", "F4Z__price__price_acceleration_rz__5m__1h__value", "F4Z__price__price_acceleration_rz__5m__4h__value", "F4Z__price__price_displacement_percentile__12h__value", "F4Z__price__price_displacement_percentile__15m__value", "F4Z__price__price_displacement_percentile__1h__value", "F4Z__price__price_displacement_percentile__4h__value", "F4Z__price__price_displacement_percentile__5m__value", "F4Z__price__price_displacement_rz__12h__value", "F4Z__price__price_displacement_rz__15m__value", "F4Z__price__price_displacement_rz__1h__value", "F4Z__price__price_displacement_rz__4h__value", "F4Z__price__price_displacement_rz__5m__value", "F4Z__price__price_excursion_range_percentile__12h__value", "F4Z__price__price_excursion_range_percentile__15m__value", "F4Z__price__price_excursion_range_percentile__1h__value", "F4Z__price__price_excursion_range_percentile__4h__value", "F4Z__price__price_excursion_range_percentile__5m__value", "F4Z__price__price_excursion_range_rz__12h__value", "F4Z__price__price_excursion_range_rz__15m__value", "F4Z__price__price_excursion_range_rz__1h__value", "F4Z__price__price_excursion_range_rz__4h__value", "F4Z__price__price_excursion_range_rz__5m__value", "F4Z__price__price_move_retention_percentile__12h__value", "F4Z__price__price_move_retention_percentile__15m__value", "F4Z__price__price_move_retention_percentile__1h__value", "F4Z__price__price_move_retention_percentile__4h__value", "F4Z__price__price_move_retention_percentile__5m__value", "F4Z__volatility__range_expansion_percentile__15m__12h__value", "F4Z__volatility__range_expansion_percentile__15m__1h__value", "F4Z__volatility__range_expansion_percentile__15m__4h__value", "F4Z__volatility__range_expansion_percentile__1h__12h__value", "F4Z__volatility__range_expansion_percentile__1h__4h__value", "F4Z__volatility__range_expansion_percentile__4h__12h__value", "F4Z__volatility__range_expansion_percentile__5m__12h__value", "F4Z__volatility__range_expansion_percentile__5m__15m__value", "F4Z__volatility__range_expansion_percentile__5m__1h__value", "F4Z__volatility__range_expansion_percentile__5m__4h__value", "F4Z__volatility__range_expansion_rz__15m__12h__value", "F4Z__volatility__range_expansion_rz__15m__1h__value", "F4Z__volatility__range_expansion_rz__15m__4h__value", "F4Z__volatility__range_expansion_rz__1h__12h__value", "F4Z__volatility__range_expansion_rz__1h__4h__value", "F4Z__volatility__range_expansion_rz__4h__12h__value", "F4Z__volatility__range_expansion_rz__5m__12h__value", "F4Z__volatility__range_expansion_rz__5m__15m__value", "F4Z__volatility__range_expansion_rz__5m__1h__value", "F4Z__volatility__range_expansion_rz__5m__4h__value", "F4Z__volatility__realized_variance_level_percentile__12h__value", "F4Z__volatility__realized_variance_level_percentile__15m__value", "F4Z__volatility__realized_variance_level_percentile__1h__value", "F4Z__volatility__realized_variance_level_percentile__4h__value", "F4Z__volatility__realized_variance_level_percentile__5m__value", "F4Z__volatility__realized_variance_level_rz__12h__value", "F4Z__volatility__realized_variance_level_rz__15m__value", "F4Z__volatility__realized_variance_level_rz__1h__value", "F4Z__volatility__realized_variance_level_rz__4h__value", "F4Z__volatility__realized_variance_level_rz__5m__value", "F4Z__volatility__vol_acceleration_percentile__15m__12h__value", "F4Z__volatility__vol_acceleration_percentile__15m__1h__value", "F4Z__volatility__vol_acceleration_percentile__15m__4h__value", "F4Z__volatility__vol_acceleration_percentile__1h__12h__value", "F4Z__volatility__vol_acceleration_percentile__1h__4h__value", "F4Z__volatility__vol_acceleration_percentile__4h__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__12h__value", "F4Z__volatility__vol_acceleration_percentile__5m__15m__value", "F4Z__volatility__vol_acceleration_percentile__5m__1h__value", "F4Z__volatility__vol_acceleration_percentile__5m__4h__value", "F4Z__volatility__vol_acceleration_rz__15m__12h__value", "F4Z__volatility__vol_acceleration_rz__15m__1h__value", "F4Z__volatility__vol_acceleration_rz__15m__4h__value", "F4Z__volatility__vol_acceleration_rz__1h__12h__value", "F4Z__volatility__vol_acceleration_rz__1h__4h__value", "F4Z__volatility__vol_acceleration_rz__4h__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__12h__value", "F4Z__volatility__vol_acceleration_rz__5m__15m__value", "F4Z__volatility__vol_acceleration_rz__5m__1h__value", "F4Z__volatility__vol_acceleration_rz__5m__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__12h__value", "F4Z__volatility__vol_shock_concentration_percentile__15m__value", "F4Z__volatility__vol_shock_concentration_percentile__1h__value", "F4Z__volatility__vol_shock_concentration_percentile__4h__value", "F4Z__volatility__vol_shock_concentration_percentile__5m__value", "F4Z__volatility__vol_shock_recency_percentile__12h__value", "F4Z__volatility__vol_shock_recency_percentile__15m__value", "F4Z__volatility__vol_shock_recency_percentile__1h__value", "F4Z__volatility__vol_shock_recency_percentile__4h__value", "F4Z__volatility__vol_shock_recency_percentile__5m__value"], "pilot_features": ["F4Z__flow__flow_activity_percentile__5m__value", "F4Z__flow__flow_pressure_rz__1h__value", "F4Z__flow__cvd_path_efficiency_percentile__4h__value"], "pilot_horizons": [15, 2880, 18000], "horizons": [15, 60, 240, 480, 960, 2880, 4320, 7200, 12000, 18000], "months": ["2022-04", "2022-05", "2022-06", "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12", "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06", "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"], "origin_end_exclusive": "2024-07-01T00:00:00Z", "reserve_start": "2024-07-24T00:01:00Z", "cadence_minutes": 30, "folds": [["2023-04-01", "2023-07-01"], ["2023-07-01", "2023-10-01"], ["2023-10-01", "2024-01-01"], ["2024-01-01", "2024-04-01"], ["2024-04-01", "2024-07-01"]], "grid": [[2, 64], [2, 256], [3, 64], [3, 256], [5, 64], [5, 256]], "minimum_observed_training": 1000, "inner_validation": "Preceding three calendar months; inner training label_end < inner start, inner validation label_end < outer start. Outer training label_end < outer start.", "model": "Single raw F4Z input plus explicit missing indicator; DecisionTreeRegressor absolute_error, seed48, depth/leaf grid only.", "preprocessing": "Train-only finite-value median imputation plus missing indicator. No feature scaling. Baseline_n >=1000, finite, valid, recorded available_after <= origin, scientific_status PROVISIONAL explicitly allowed for provisional research only.", "selection": "Lowest inner MAE; ties shallower depth then larger leaf. Never use outer MAE to choose fold settings.", "ineligible_policy": "If fewer than1000 observed training values or constant observed training feature: no fit; if no eligible inner setting or outer training insufficient, predict outer-training target median on all outer rows. Report fallback and coverage.", "evaluation": "Exact original baseline rows; primary MAE minutes, secondary percent of horizon; pooled row-weighted errors; compare all three baselines. Record missing/observed subset errors. No significance/global optimum/trading claim.", "pilot_scope": "3 flow features x3 horizons x5 folds x(6 inner+1 outer) = at most315 fits. Features selected for differing market meanings and sampled coverage, not outcome scores.", "pilot_meanings": {"F4Z__flow__flow_activity_percentile__5m__value": "Recent flow activity relative to history", "F4Z__flow__flow_pressure_rz__1h__value": "Directional trading pressure relative to history", "F4Z__flow__cvd_path_efficiency_percentile__4h__value": "How consistently cumulative trading flow follows one direction"}, "cache_policy": "One compact research-only NPZ per horizon/family in Drive, no PC cache. Float64 values preserve source conversion. Checkpoint each month; immutable signature/hash checks. Family cache includes all candidate columns from that family (85 flow) to avoid rereading later. <=128MiB per cache.", "checkpoint_policy": "Per-fit compact JSON scores in Timing_Screen_Fit_Scores; no model binaries or row predictions. Partial family/horizon data resume completed months. An interrupted unsaved fit may restart. Pilot completed fits reused by full mode under identical identity/versions.", "workers": "Up to4 bounded I/O workers; up to2 fit workers; native thread pools limited. No automatic full run after pilot.", "limitations": ["F4Z upstream timing/normalization remains unresolved; PROVISIONAL research only.", "Sample-based candidate preselection does not prove excluded features useless across full history.", "Single-feature trees do not establish globally best features or combinations.", "Reserved period previously exposed in other project work but excluded from this task.", "Errors overlap in time; no iid significance assumptions."], "full_launch_policy": "Review pilot correctness, runtime, memory and results before authorizing full386-feature expansion. Do not change to full mode yet."}, "sources": {"X": {"id": "1RXZDR_PaLkKMYtuswOBnl4_0eG9_LkNx", "name": "Handoff 1_F4_f4Z.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "043b2f570cfb63215749385d72c7fe59", "size": "15600742757"}, "Y": {"id": "1y351d1ibxpAH-MxijvGDHPWEfqOJ70aM", "name": "HANDOFF_2_CANONICAL_XY_V1_1.zip", "modifiedTime": "2026-11-13T05:57:28.270Z", "md5Checksum": "bf2cb8dd274222747fde1b2e979b42b7", "size": "3033662926"}}, "runner_sha256": "25ae53d6a34e1695381372f46bcd26298428c341517336f3096e6d9815b172f0", "versions": {"python": "3.13.18", "numpy": "2.4.6", "pandas": "2.2.3", "sklearn": "1.6.1", "pyarrow": "23.0.1"}}')
WORKER["run_check"](config,pilot_identity,sources,hashes)